# 72 · Validation of the package coordinate

**What this does.** The user adopted Caleb's package `pseudospace_reconstruction` (stages 01–04) as the method that
builds the pseudospace. This notebook asks whether its coordinate measures position along the PT, and only
position. It reruns, on the package's units, the checks notebooks 35, 36, 51–53 and 60 ran on SCF13:
- external agreement (P1, P1w);
- within-segment order (gene folds, LOSO);
- robustness to gene folds, count splitting, equal depth and an injected group effect;
- positional noise.

Each refit reruns the package itself, unmodified, on perturbed stage-02 inputs: stages 03–04 through its command
line. The criteria are validation criteria. Each is reported as met or not met with the claim it allows; none can
un-adopt the package.

**The frozen protocol.** `docs/results/pt-package-coordinate-validation.md`, written before any fit it governs.

**Inputs.** Notebook 70's export and run folders (`results/package_pseudospace/`); external segment references
(`data/external/`); notebook 71's tables (read only, for C10 and the 71 checks); cached positions of SCF13, DPT13,
LCP and LRS-Z (section 6 only).

**Outputs** (`results/pt_package_coordinate_validation/`): `tables/`, `figures/`, `refits/{hvm,aki}/<refit>/`
(manifests, labels and positions; large files deleted after extraction), `stage_cache/`, `run_record.json`.

In [ ]:
import os

# Resource caps (notebook 72 shares the machine): at most 2 package refits at once; everything else in this kernel is
# capped at about 24 threads. The refits themselves run with the libraries' default thread counts, as notebook 70 ran:
# the package's stage 03 changes its clusters when only the BLAS thread count changes (section 4). Never in a cache key.
for _name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ.setdefault(_name, '24')

import argparse
import json
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from hashlib import sha256
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--visium-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser().resolve()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser().resolve()
visium_root = args.visium_root or os.environ.get('PSEUDOSPACE_VISIUM_ROOT')
if not visium_root:
    raise RuntimeError('The package configs expand $PSEUDOSPACE_VISIUM_ROOT: set it or pass --visium-root (notebook 70).')
os.environ['PSEUDOSPACE_VISIUM_ROOT'] = str(Path(visium_root).expanduser().resolve())

NOTEBOOK_LOGIC_VERSION = '72.package_coordinate_validation.1'   # Bump when a cached calculation changes.
REFIT_LOGIC_VERSION = '72.refits.1'          # Part of the refit cache key (with package_refits.py).
EXTERNAL_VERSION, EXTERNAL_COMMIT, HARMONY_VERSION = '0.1.0', '59da04b', '2.0.5'
CONFIRMED_PASS1 = {'hvm': 'efafc9ac2e99', 'aki': 'bb4a80c1b484'}   # coordinator's label checkpoint (notebook 70)
DATASETS = {'hvm': 'human_vs_mouse', 'aki': 'aki'}                  # dataset -> tracked config folder
CONFIGS = {ds: project / 'analysis' / 'package_configs' / folder for ds, folder in DATASETS.items()}
package_out = results_root / 'package_pseudospace'
RUNS = {ds: package_out / 'runs' / ds for ds in DATASETS}
EXPORTS = {ds: package_out / ds / 'pt_units.h5ad' for ds in DATASETS}
GROUPS = {'hvm': {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']},     # reference first
          'aki': {'control': ['Ctrl1A2', 'Ctrl1A4'], 'IR': ['IR2A2', 'IR2A4']}}
GROUP_OF_LABEL = {'Mouse': 'mouse', 'Human': 'human', 'Control': 'control', 'IR': 'IR'}
SEGMENTS = ['PT-S1', 'PT-S2', 'PT-S3']
N_FOLDS = 3
MIN_DETECTION = .10                          # notebook 35: per species (group), in our PT units
ANCHOR_LOOSE, ANCHOR_STRICT = .5, 1.0
INJECTION_SHARE_MOD, INJECTION_SEED = 10, 351
ABSORPTION_FLOOR = .02
EQUAL_DEPTH_QUANTILE, EQUAL_DEPTH_SEED = .20, 352
SPLIT_EPSILON, SPLIT_SEED = .5, 35
BASIS_DF, P4C_DF = 6, 4
DF1, CAMERA_COR, ALPHA = 6, .01, .05         # notebook 69/71's positional F-test and cameraPR
REFIT_JOBS = int(os.environ.get('PSEUDOSPACE_REFIT_JOBS', 2))
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', 24))
MIN_FREE_GB = 40.
output = results_root / 'pt_package_coordinate_validation'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
REFIT_DIR = output / 'refits'
for folder in (tables, figures, cache, REFIT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
nb71 = results_root / 'pt_pathways_package_pseudospace'
external = data_root / 'external'
READING = []    # how the protocol was read where it is silent (reported in run_record.json and the summary)
print('Results folder:', output)

## 1 · Guards (not cached)

- The package is 0.1.0 at 59da04b and R harmony is 2.0.5.
- Notebook 70's pass-1 fingerprints equal the ones the coordinator confirmed at the label checkpoint (hvm
  `efafc9ac2e99`, AKI `bb4a80c1b484`), in the run manifests and in the export provenance.
- Every export unit is a pass-2 unit of its run, with the same pseudospace.

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from importlib.metadata import distribution, version
from IPython.display import display
from scipy import sparse
from scipy.stats import spearmanr

INJECTION_LOG_AMPLITUDE = float(np.log(2))   # f(s) = log 2 * (2s - 1): 0.5x to 2x along the PT (notebook 35)
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 200)
direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown')}
assert EXTERNAL['version'] == EXTERNAL_VERSION and EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), EXTERNAL
harmony = subprocess.run([sys.executable, '-c', (
    'import os, sys; os.environ.setdefault("R_HOME", os.path.join(sys.prefix, "lib", "R")); '
    'import rpy2.robjects as ro; print(ro.r("as.character(packageVersion(\'harmony\'))")[0])')],
    capture_output=True, text=True).stdout.strip().splitlines()
assert harmony and harmony[-1] == HARMONY_VERSION, f'R harmony must be {HARMONY_VERSION}: {harmony}'

from pseudospace_reconstruction import config as pcfg, pipeline

import pseudospace.package_refits as refit_module
from pseudospace.package_inputs import read_pt_units, support_positions
from pseudospace.stage_cache import cached_frame, cached_payload, digest

MANIFEST = {ds: {k: json.loads((RUNS[ds] / folder / 'manifest.json').read_text())
                 for k, folder in (('02', '02_qc'), ('03', '03_integration'), ('04', '04_pseudospace'))}
            for ds in DATASETS}
UNITS = {}
for ds in DATASETS:
    run_fp = MANIFEST[ds]['03']['summary']['fingerprint_pass1']['membership_sha1']
    UNITS[ds] = read_pt_units(EXPORTS[ds], ds)
    export_fp = UNITS[ds].uns['provenance']['fingerprints']['pass1']['membership_sha1']
    assert run_fp == export_fp == CONFIRMED_PASS1[ds], (ds, run_fp, export_fp, CONFIRMED_PASS1[ds])
    table = pd.read_parquet(RUNS[ds] / '04_pseudospace' / 'pseudospace.parquet')
    assert UNITS[ds].obs_names.isin(table.index).all()
    np.testing.assert_array_equal(UNITS[ds].obs.pseudospace.to_numpy(float),
                                  table.pseudospace.reindex(UNITS[ds].obs_names).to_numpy(float))
DATASET_CONFIG = {ds: pcfg.load_config(CONFIGS[ds], data_root) for ds in DATASETS}
PANELS = yaml.safe_load((CONFIGS['hvm'] / 'marker_panels.yaml').read_text())
assert PANELS == yaml.safe_load((CONFIGS['aki'] / 'marker_panels.yaml').read_text())
FIXED_GENES = sorted({g for s in SEGMENTS for g in PANELS['label_panels'][s]}
                     | set(PANELS['axis_anchors']['early']) | set(PANELS['axis_anchors']['late']))
assert len(FIXED_GENES) == 15, FIXED_GENES
print('guards passed:', EXTERNAL, '· confirmed pass-1 fingerprints', CONFIRMED_PASS1)
print('Genes that set labels or orientation (never evaluation genes):', ', '.join(FIXED_GENES))

## 2 · Units, positions and expression

- **Units:** each dataset's PT export, all units; segments are the package's `segment_class` (base labels for
  every refit, as notebook 35 kept its reviewed labels).
- **Position:** the raw stage-04 value for rank metrics. Scale-dependent metrics (noise, gap, injection) use
  notebook 71's rule applied to that coordinate: min–max over the units inside each specimen's [1st, 99th] common
  support (`support_positions`), extended affinely to every unit.
- **Expression for P1/P1w/P4:** notebook 35's log1p(count / full library × 10⁴) (decision O3).

In [ ]:
from pseudospace.coordinate_evaluation import log_normalize


def scale71(raw, specimen, segment):
    """Notebook 71's rule on any coordinate: min-max over the common-support units, applied affinely to all."""
    raw = np.asarray(raw, float)
    out = np.full(len(raw), np.nan)
    finite = np.isfinite(raw)
    mask, kept = support_positions(raw[finite], np.asarray(specimen)[finite], np.asarray(segment)[finite])
    lo, hi = raw[finite][mask].min(), raw[finite][mask].max()
    out[finite] = (raw[finite] - lo) / (hi - lo)
    return out


D = {}
for ds, units in UNITS.items():
    obs = units.obs
    d = {'ids': pd.Index(units.obs_names.astype(str)), 'specimen': obs['sample'].astype(str).to_numpy(),
         'group': obs['group'].astype(str).map(GROUP_OF_LABEL).to_numpy(), 'segment': obs.segment_class.astype(str).to_numpy(),
         'raw': obs.pseudospace.to_numpy(float), 'dpt': obs.dpt_pseudotime.to_numpy(float),
         'genes': pd.Index(units.var_names.astype(str)), 'counts': sparse.csr_matrix(units.X)}
    d['groups'] = GROUPS[ds]
    d['ref'], d['case_label'] = list(GROUPS[ds])
    d['case'] = d['group'] == d['case_label']
    d['seg'] = pd.Series(d['segment']).str.replace('PT-', '', regex=False).to_numpy()
    d['pos71'] = scale71(d['raw'], d['specimen'], d['segment'])
    d['support'] = (d['pos71'] >= 0) & (d['pos71'] <= 1)
    d['library'] = np.asarray(d['counts'].sum(axis=1)).ravel().astype(float)   # every export gene (all measured)
    d['Y'] = log_normalize(d['counts'], d['library'])
    d['log_library'] = np.log(d['library'])
    d['n_spots'] = obs.n_spots.to_numpy(float)
    d['n_detected'] = np.asarray((d['counts'] > 0).sum(axis=1)).ravel().astype(float)
    d['detection'] = pd.DataFrame({g: np.asarray((d['counts'][d['group'] == g] > 0).mean(axis=0)).ravel()
                                   for g in GROUPS[ds]}, index=d['genes'])
    d['measured'] = pd.Series(np.asarray(d['counts'].sum(axis=0)).ravel() > 0, index=d['genes'])
    D[ds] = d
    print(f'{ds}: {len(d["ids"]):,} PT units ({int(d["support"].sum()):,} in the common support) · {len(d["genes"]):,} genes')
READING.append('Library for P1/P1w/P4 and for equal depth: the total over every export gene (all are measured; in '
               'human vs mouse every gene is a shared ortholog, matching notebook 35\'s measured-in-both total).')

## 3 · External references and gene sets (notebook 35's rules on the package units)

Anchors are selected on mouse microdissection, female mouse snRNA and human snRNA donor half A; everything is
evaluated on male mouse snRNA (with its adjacent contrasts for P1w) and human snRNA donor half B. Microdissection
is reported for continuity only (it selected the LCP and LRS-Z landmarks). PT-specificity uses the class means of
the package's non-PT families (pass-2 units of the base run).

In [ ]:
from pseudospace.anchor_coordinate import (bulk_segment_profiles, census_segment_profiles, conserved_anchor_mask,
                                           positive_control_masks, pt_specific_mask, segment_contrast)
from pseudospace.coordinate_evaluation import sha256_folds

genes_union = D['hvm']['genes'].union(D['aki']['genes'])
mouse_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
mouse_micro = bulk_segment_profiles(mouse_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol'),
                                    '_PT{seg}_').reindex(genes_union)


def census(label, segment_map, early):
    counts = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    return census_segment_profiles(counts, meta.loc[counts.index], segment_map, early=early).reindex(genes_union)


mouse_sn = census('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                            'epithelial cell of proximal tubule segment 2': 'S2',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
human_sn = census('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, None)
human_donors = sorted(set(human_sn.columns.get_level_values('donor')))
HALF_A, HALF_B = human_donors[0::2], human_donors[1::2]
if HALF_A != ['21_015', 'HCA_51_ref', 'PRECISE-34'] or HALF_B != ['HCA_29_ref', 'HCA_55_ref', 'PRECISE-36']:
    raise ValueError(f'Human donor halves differ from the pre-registered split: {HALF_A} / {HALF_B}')


def micro_contrast(frame, late, early):
    return frame[late] - frame[early], frame[[late, early]].max(axis=1)


ext = {
    'select_mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
    'select_mouse_sn_female': segment_contrast(mouse_sn, sex='female'),
    'select_human_sn_halfA': segment_contrast(human_sn, donors=HALF_A),
    'mouse_sn_male': segment_contrast(mouse_sn, sex='male'),
    'mouse_sn_male_S2mS1': segment_contrast(mouse_sn, late='S2', early='S1', sex='male'),
    'mouse_sn_male_S3mS1': segment_contrast(mouse_sn, late='S3', early='S1', sex='male'),
    'mouse_sn_male_S3mS2': segment_contrast(mouse_sn, late='S3', early='S2', sex='male'),
    'mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
    'mouse_micro_S2mS1': micro_contrast(mouse_micro, 'S2', 'S1'),
    'mouse_micro_S3mS1': micro_contrast(mouse_micro, 'S3', 'S1'),
    'mouse_micro_S3mS2': micro_contrast(mouse_micro, 'S3', 'S2'),
    'human_sn_halfB': segment_contrast(human_sn, donors=HALF_B),
}
contrast = pd.DataFrame({k: v[0] for k, v in ext.items()})
level = pd.DataFrame({k: v[1] for k, v in ext.items()})

# PT-specificity (notebook 35): linear-scale class means per specimen (>= 20 units), averaged within species,
# over the base run's pass-2 units and families.
h = D['hvm']
base_qc = ad.read_h5ad(RUNS['hvm'] / '02_qc' / 'units_qc.h5ad', backed='r')
base_table = pd.read_parquet(RUNS['hvm'] / '04_pseudospace' / 'pseudospace.parquet')
rows = base_qc.obs_names.get_indexer(base_table.index)
assert (rows >= 0).all()
neph_counts = sparse.csr_matrix(base_qc.layers['counts'][np.sort(rows)])
neph_names = base_qc.obs_names[np.sort(rows)]
base_qc.file.close()
neph_obs = base_table.loc[neph_names]
assert pd.Index(base_qc.var_names).equals(h['genes'])
neph_library = np.asarray(neph_counts.sum(axis=1)).ravel().astype(float)
normalized = sparse.csr_matrix(neph_counts.multiply((1e4 / neph_library)[:, None]))
classes = neph_obs.coarse_class.astype(str).to_numpy()
specimen_neph = neph_obs['sample'].astype(str).to_numpy()
class_means = {}
for sp, names in GROUPS['hvm'].items():
    columns = {}
    for cls in sorted(set(classes)):
        per = [np.asarray(normalized[(classes == cls) & (specimen_neph == n)].mean(axis=0)).ravel()
               for n in names if ((classes == cls) & (specimen_neph == n)).sum() >= 20]
        if per:
            columns[cls] = np.mean(per, axis=0)
    class_means[sp] = pd.DataFrame(columns, index=h['genes'])
del normalized, neph_counts
measured_both = pd.Series(True, index=h['genes'])     # every export gene is a shared ortholog measured in both
eligible_anchor = (measured_both & h['detection'].mouse.ge(MIN_DETECTION) & h['detection'].human.ge(MIN_DETECTION)
                   & pt_specific_mask(class_means))
selection = ['select_mouse_micro', 'select_mouse_sn_female', 'select_human_sn_halfA']
c_h, l_h = contrast.reindex(h['genes']), level.reindex(h['genes'])
A_LOOSE = conserved_anchor_mask(c_h[selection], l_h[selection], eligible_anchor, threshold=ANCHOR_LOOSE)
A_STRICT = conserved_anchor_mask(c_h[selection], l_h[selection], eligible_anchor, threshold=ANCHOR_STRICT)
anchor_sign = np.sign(contrast['select_human_sn_halfA'])
fixed = pd.Series(h['genes'].isin(FIXED_GENES), index=h['genes'])
control_eligible = (measured_both & ~A_LOOSE & ~fixed & h['detection'].mouse.ge(MIN_DETECTION)
                    & h['detection'].human.ge(MIN_DETECTION) & l_h['mouse_sn_male'].ge(1))
D_OPPOSITE, D_FLAT = positive_control_masks(c_h['human_sn_halfB'], c_h['mouse_sn_male'], c_h['mouse_micro'],
                                            l_h['human_sn_halfB'], control_eligible)
BASE_FEATURES = {}
for ds in DATASETS:
    feats = set()
    for name in ('pass1_labelled.h5ad', 'pass2.h5ad'):
        obj = ad.read_h5ad(RUNS[ds] / '03_integration' / name, backed='r')
        feats |= set(obj.var_names[obj.var['highly_variable_for_harmony'].to_numpy(bool)].astype(str))
        if name == 'pass2.h5ad':
            BASE_FEATURES[(ds, 'pass2')] = sorted(obj.var_names[obj.var['highly_variable_for_harmony'].to_numpy(bool)].astype(str))
        obj.file.close()
    BASE_FEATURES[ds] = sorted(feats)
H_SET = A_STRICT & ~pd.Series(h['genes'].isin(BASE_FEATURES['hvm']), index=h['genes'])
A_LOOSE_ALL = A_LOOSE.reindex(genes_union, fill_value=False)


def evaluation_mask(ds, group, reference):
    """Notebook 35's E: measured, not an anchor, not a label/orientation gene, detected >= 10% in the group, external
    level >= 1 and a finite reference contrast."""
    d = D[ds]
    genes = d['genes']
    return (d['measured'] & ~A_LOOSE_ALL.reindex(genes).astype(bool) & ~pd.Series(genes.isin(FIXED_GENES), index=genes)
            & d['detection'][group].ge(MIN_DETECTION) & level[reference].reindex(genes).ge(1)
            & contrast[reference].reindex(genes).notna())


for ds, d in D.items():
    masks = [evaluation_mask(ds, g, ref) for g in d['groups']
             for ref in (('human_sn_halfB',) if g == 'human' else ('mouse_sn_male', 'mouse_micro'))]
    eval_mask = pd.concat(masks, axis=1).any(axis=1)
    if ds == 'hvm':
        eval_mask |= D_OPPOSITE | D_FLAT | H_SET
    d['eval_genes'] = d['genes'][eval_mask.to_numpy()]
    d['eval_cols'] = d['genes'].get_indexer(d['eval_genes'])
    d['EX'] = d['Y'][:, d['eval_cols']]
    d['fold'] = pd.Series(sha256_folds(d['genes'], N_FOLDS), index=d['genes'])
    d['eval_fold'] = d['fold'].loc[d['eval_genes']].to_numpy()

old_sets = pd.read_csv(results_root / 'pt_reconstruction_v2' / 'nb35' / 'gene_sets_and_external_contrasts.csv', index_col=0)


def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(len(a | b), 1)


SET_COUNTS = pd.DataFrame([
    {'set': 'anchors |Δ| ≥ 0.5 (A_LOOSE)', 'genes': int(A_LOOSE.sum()),
     'Jaccard with notebook 35': jaccard(A_LOOSE.index[A_LOOSE], old_sets.index[old_sets.anchor_loose_A166rule])},
    {'set': 'anchors |Δ| ≥ 1 (A_STRICT)', 'genes': int(A_STRICT.sum()),
     'Jaccard with notebook 35': jaccard(A_STRICT.index[A_STRICT], old_sets.index[old_sets.anchor_strict_A49rule])},
    {'set': 'positive controls, opposite sign', 'genes': int(D_OPPOSITE.sum()),
     'Jaccard with notebook 35': jaccard(D_OPPOSITE.index[D_OPPOSITE], old_sets.index[old_sets.positive_opposite])},
    {'set': 'positive controls, mouse-zonated human-flat', 'genes': int(D_FLAT.sum()),
     'Jaccard with notebook 35': jaccard(D_FLAT.index[D_FLAT], old_sets.index[old_sets.positive_flat])},
    {'set': 'conserved held-out H', 'genes': int(H_SET.sum()),
     'Jaccard with notebook 35': jaccard(H_SET.index[H_SET], old_sets.index[old_sets.conserved_heldout_H])},
    {'set': 'base integration features, hvm (pass 1 ∪ 2)', 'genes': len(BASE_FEATURES['hvm']), 'Jaccard with notebook 35': np.nan},
    {'set': 'base integration features, AKI (pass 1 ∪ 2)', 'genes': len(BASE_FEATURES['aki']), 'Jaccard with notebook 35': np.nan},
    *[{'set': f'evaluation genes, {ds}', 'genes': len(D[ds]['eval_genes']), 'Jaccard with notebook 35': np.nan} for ds in D]])
SET_COUNTS.to_csv(tables / 'gene_set_counts.csv', index=False)
pd.DataFrame({'anchor_loose': A_LOOSE, 'anchor_strict': A_STRICT, 'positive_opposite': D_OPPOSITE,
              'positive_flat': D_FLAT, 'conserved_heldout_H': H_SET, 'fixed_label_or_orientation': fixed,
              'fold': D['hvm']['fold'], **c_h.add_prefix('contrast_'),
              **D['hvm']['detection'].add_prefix('detection_')}).to_csv(tables / 'gene_sets_hvm.csv')
display(SET_COUNTS.round(3))
READING.append('Evaluation genes also exclude the 15 genes that set the package labels or orientation (PT panels and '
               'axis anchors), and positive controls use the same exclusion (notebook 35 excluded its 6 markers).')

## 4 · Refits: the package, unmodified, on perturbed stage-02 inputs

For each refit a new run folder gets the base QC-retained units with the perturbation applied (lognorm rebuilt by
the package's own `normalize_log1p`), plus the base stage-02 manifest; then `pseudospace-recon run 03 04` runs on
it with the tracked configuration (DPT on, so each refit also refits the package DPT). Stages 01–02 (units, QC)
are held fixed. After extraction the large h5ad files are deleted.

| Refit | Perturbation (seeds from notebook 35) |
|---|---|
| identity | none (rewritten copy) — reproduction guard |
| floor | lognorm by multiplication instead of division (order 1e-16; only pass 1 reads it) |
| fold0–2 | `exclude_from_harmony_hvg` set on the sha256 fold-k genes |
| half_A, half_B | Poisson count split ε 0.5, seed 35, every unit |
| equal_depth | binomial thinning of every unit to the pooled PT 20th percentile of the library, seed 352 |
| injection | case-group PT units, 10% of the base pass-2 integration features (not anchors or label/orientation genes): × exp(±log 2 (2s − 1)), seed 351 |

**Threads.** Each refit runs with the libraries' default thread counts, as notebook 70 did. With exactly the base
input and default threads the package reproduces notebook 70 bit for bit (identity guard); with 12 threads it does not
(checked during development; the refit `identity_12threads`, human vs mouse only, shows it here, beyond the
protocol).

**Package limits (accepted, decision O4).** QC and the unit set are fixed across refits. The package builds its
clustering feature space from all genes, so fold-k genes still shape the labels, the pass-1 drops and the root
cluster in a fold-k refit; they do not shape the embedding.

In [ ]:
from pseudospace.coordinate_evaluation import binomial_thin_to_depth, poisson_count_split, scale_counts


def hash_int(gene):
    return int(sha256(str(gene).encode()).hexdigest(), 16)


BASE = {}
SPECS = {}
for ds in DATASETS:
    base = ad.read_h5ad(RUNS[ds] / '02_qc' / 'units_qc.h5ad')
    d = D[ds]
    pt_rows = base.obs_names.get_indexer(d['ids'])
    assert (pt_rows >= 0).all()
    counts = sparse.csr_matrix(base.layers['counts']).astype(np.int64)
    np.testing.assert_array_equal(np.asarray(counts[pt_rows].sum(axis=1)).ravel(), d['library'])
    BASE[ds] = {'units': base, 'pt_rows': pt_rows, 'counts': counts}
    folds = pd.Series(sha256_folds(base.var_names.astype(str), N_FOLDS), index=base.var_names.astype(str))
    injected = sorted([g for g in BASE_FEATURES[(ds, 'pass2')] if not A_LOOSE_ALL.get(g, False)
                       and g not in FIXED_GENES and hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)
    library_all = np.asarray(counts.sum(axis=1)).ravel().astype(float)
    target = float(np.quantile(library_all[pt_rows], EQUAL_DEPTH_QUANTILE))
    s0 = np.clip(d['pos71'], 0, 1)
    BASE[ds].update({'folds': folds, 'injected': injected, 'signs': np.where(np.arange(len(injected)) % 2 == 0, 1., -1.),
                     'library_all': library_all, 'depth_target': target, 's0': s0})
    SPECS[ds] = {'identity': {'perturbation': 'none'},
                 'floor': {'perturbation': 'lognorm by multiplication'},
                 **{f'fold{k}': {'perturbation': 'withhold fold', 'fold': k, 'n_genes': int((folds == k).sum()),
                                 'genes': digest(sorted(folds.index[folds == k]))} for k in range(N_FOLDS)},
                 **{f'half_{h_}': {'perturbation': 'count split', 'epsilon': SPLIT_EPSILON, 'seed': SPLIT_SEED, 'half': h_}
                    for h_ in 'AB'},
                 'equal_depth': {'perturbation': 'equal depth', 'quantile': EQUAL_DEPTH_QUANTILE, 'seed': EQUAL_DEPTH_SEED,
                                 'target': target},
                 'injection': {'perturbation': 'injection', 'group': d['case_label'], 'genes': injected,
                               'amplitude': INJECTION_LOG_AMPLITUDE, 'seed': INJECTION_SEED, 's0': digest(s0)}}
    if ds == 'hvm':   # beyond the protocol, descriptive: the identity copy run with 12 BLAS/OpenMP threads
        SPECS[ds]['identity_12threads'] = {'perturbation': 'none', 'threads': 12}
    print(f'{ds}: {base.n_obs:,} QC units; equal-depth target {target:,.0f} counts '
          f'({np.mean(library_all[pt_rows] > target):.0%} of PT thinned); {len(injected)} injected genes')


def perturbed_counts(ds, name):
    """(counts or None, exclude_genes, multiply) for one refit."""
    b, spec = BASE[ds], SPECS[ds][name]
    kind = spec['perturbation']
    if kind == 'none':
        return None, (), False
    if kind == 'lognorm by multiplication':
        return None, (), True
    if kind == 'withhold fold':
        return None, list(b['folds'].index[b['folds'] == spec['fold']]), False
    if kind == 'count split':
        half_a, half_b = poisson_count_split(b['counts'], SPLIT_EPSILON, np.random.default_rng(SPLIT_SEED))
        return (half_a if spec['half'] == 'A' else half_b), (), False
    if kind == 'equal depth':
        return binomial_thin_to_depth(b['counts'], b['library_all'], b['depth_target'],
                                      np.random.default_rng(EQUAL_DEPTH_SEED)), (), False
    if kind == 'injection':
        d = D[ds]
        rows = b['pt_rows'][d['case']]
        f = INJECTION_LOG_AMPLITUDE * (2 * b['s0'][d['case']] - 1)
        cols = b['units'].var_names.get_indexer(b['injected'])
        return scale_counts(b['counts'], rows, cols, f[:, None] * b['signs'][None, :],
                            np.random.default_rng(INJECTION_SEED)), (), False
    raise ValueError(kind)


BASE_KEY = {ds: {'units_qc': RUNS[ds] / '02_qc' / 'units_qc.h5ad', 'manifest_02': RUNS[ds] / '02_qc' / 'manifest.json',
                 'manifest_03': RUNS[ds] / '03_integration' / 'manifest.json',
                 'config': [CONFIGS[ds] / f for f in ('samples.csv', 'dataset.yaml', 'marker_panels.yaml')],
                 'package': EXTERNAL} for ds in DATASETS}
REFIT_CODE = digest([REFIT_LOGIC_VERSION, Path(refit_module.__file__)])


def refit(ds, name):
    def compute():
        free = refit_module.free_gb(output)
        if free < MIN_FREE_GB:
            raise RuntimeError(f'Only {free:.0f} GB free; a refit needs >= {MIN_FREE_GB:.0f} GB (protocol).')
        folder = REFIT_DIR / ds / name
        if folder.exists():
            import shutil
            shutil.rmtree(folder)            # a stale, unfinished refit of this name
        t0 = time.perf_counter()
        counts, exclude, multiply = perturbed_counts(ds, name)
        units = refit_module.perturbed_units(BASE[ds]['units'], counts=counts, exclude_genes=exclude, multiply=multiply)
        del counts
        refit_module.write_refit_input(units, RUNS[ds], folder, {'name': name, **SPECS[ds][name],
                                                                 'notebook': NOTEBOOK_LOGIC_VERSION})
        del units
        refit_module.run_stages_03_04(CONFIGS[ds], data_root, folder, threads=SPECS[ds][name].get('threads'))
        payload = refit_module.extract_refit(folder)
        payload['freed_bytes'] = np.array(refit_module.cleanup_refit(folder))
        payload['seconds'] = np.array(time.perf_counter() - t0)
        return payload
    return cached_payload(f'refit_{ds}_{name}', compute, root=cache, params={'name': name, **SPECS[ds][name]},
                          inputs=BASE_KEY[ds], code=REFIT_CODE)


jobs = [(ds, name) for ds in DATASETS for name in SPECS[ds]]
with ThreadPoolExecutor(max_workers=REFIT_JOBS) as pool:
    REFITS = dict(zip(jobs, pool.map(lambda job: refit(*job), jobs)))
for ds in DATASETS:
    BASE[ds].pop('counts', None)
READING.append('Refits run with the libraries\' default thread counts, as notebook 70 ran. With exactly the base input this '
               'reproduces notebook 70 bit for bit. During development a 12-thread run of the same input changed the pass-1 '
               'clustering and moved PT units (identity_12threads repeats it here, beyond the protocol).')
READING.append('The perturbed stage-02 file keeps the package\'s own layout (lognorm with sorted column indices). During '
               'development an unsorted lognorm with identical values alone changed the stage-03 clusters.')
print(pd.DataFrame([{'dataset': ds, 'refit': name, 'seconds': float(p.get('seconds', np.nan)),
                     'pass-2 units': len(p['unit'])} for (ds, name), p in REFITS.items()]).to_string())

## 5 · Refit guards and positions (not cached)

- **Identity:** the rewritten copy must reproduce notebook 70 (Spearman ≥ 0.999 over every pass-2 unit, identical
  pass-1 fingerprint); otherwise the notebook stops.
- **Folds:** no fold-k gene may be among a fold-k refit's integration features (pass 1 or pass 2).
- **Orientation:** a refit whose PT-S1 median is not below its PT-S3 median in some specimen counts as failed; it is
  never repaired and its positions are NaN (notebook 36).
- Recorded per refit: base PT units missing from pass 2 (**new:** > 5% flagged as unit-set instability), PT family
  retention and S1/S2/S3 label agreement (κ) with the base, the root and the fingerprints.

Package DPT gets notebook 70's flip rule in every refit.

In [ ]:
from sklearn.metrics import cohen_kappa_score

COORD = {ds: {'package': {}, 'package DPT': {}} for ds in DATASETS}
RECORD_ROWS = []
for ds, d in D.items():
    base_table = pd.read_parquet(RUNS[ds] / '04_pseudospace' / 'pseudospace.parquet')
    COORD[ds]['package']['full'] = d['raw']
    COORD[ds]['package DPT']['full'] = d['dpt']
    for name in SPECS[ds]:
        p = REFITS[(ds, name)]
        m3 = json.loads(str(p['manifest_03']))
        z = refit_module.refit_position(p, d['ids'])
        dpt, flipped = refit_module.orient_like_export(refit_module.refit_position(p, d['ids'], 'dpt_pseudotime'),
                                                       d['segment'])
        pass1 = pd.DataFrame({'segment': p['pass1_segment_class'], 'family': p['pass1_coarse_class']},
                             index=pd.Index(p['pass1_unit'].astype(str)))
        lab = pass1.reindex(d['ids'])
        both_pt = lab.family.eq('PT').to_numpy()
        row = {'dataset': ds, 'refit': name, 'pass-2 units': len(p['unit']),
               'base PT units missing from pass 2': float(np.mean(~np.isfinite(z))),
               'PT family retained': float(both_pt.mean()),
               'κ S1/S2/S3 vs base': float(cohen_kappa_score(d['segment'][both_pt], lab.segment.to_numpy()[both_pt])),
               'pass-1 fingerprint': m3['fingerprint_pass1']['membership_sha1'],
               'pass-1 clusters': m3['n_clusters_pass1'], 'root': json.loads(str(p['manifest_04']))['root_unit'],
               'DPT flipped': flipped, 'seconds': float(p['seconds'])}
        if name.startswith('fold'):
            k = int(name[-1])
            fold_genes = set(BASE[ds]['folds'].index[BASE[ds]['folds'] == k])
            leaked = fold_genes & (set(p['pass1_features'].astype(str)) | set(p['pass2_features'].astype(str)))
            assert not leaked, f'{ds} {name}: withheld genes among integration features: {sorted(leaked)[:5]}'
            row['withheld genes'] = len(fold_genes)
        row['integration features (pass 1 / 2)'] = f"{len(p['pass1_features'])} / {len(p['pass2_features'])}"
        row['unit-set instability (> 5% missing)'] = row['base PT units missing from pass 2'] > .05
        for arm, values in (('package', z), ('package DPT', dpt)):
            ok = refit_module.runs_s1_to_s3(values, d['specimen'], d['segment'])
            row[f'{arm} runs S1 → S3 in every specimen'] = ok
            COORD[ds][arm][name] = values if ok else np.full(len(values), np.nan)
        if name == 'identity':
            same = base_table.pseudospace.reindex(pd.Index(p['unit'].astype(str))).to_numpy(float)
            rho = spearmanr(same, p['pseudospace'], nan_policy='omit').statistic
            row['identity Spearman (all pass-2 units)'] = float(rho)
            row['identity max |Δ|'] = float(np.nanmax(np.abs(same - p['pseudospace'])))
            assert len(p['unit']) == len(base_table) and rho >= .999 and \
                row['pass-1 fingerprint'] == CONFIRMED_PASS1[ds], f'{ds}: the identity refit does not reproduce notebook 70'
        RECORD_ROWS.append(row)
    COORD[ds]['package']['ed'] = COORD[ds]['package'].pop('equal_depth')
    COORD[ds]['package DPT']['ed'] = COORD[ds]['package DPT'].pop('equal_depth')
REFIT_RECORD = pd.DataFrame(RECORD_ROWS)
for ds in DATASETS:
    REFIT_RECORD[REFIT_RECORD.dataset.eq(ds)].to_csv(tables / f'refit_record_{ds}.csv', index=False)
display(REFIT_RECORD.drop(columns=['root']).round(4))

## 6 · Metric machinery (notebooks 35, 36, 51 and 60, unchanged definitions)

A *context* is a set of PT units (all of a dataset's units, or the matched set M of section 13) with its specimens,
groups, segments and expression. Every metric below takes a context and the coordinates of one arm on it.

In [ ]:
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.coordinate_arms import registration_gap, within_segment_agreement
from pseudospace.coordinate_evaluation import absorption_ratio, concordance, fold_noise_variance, group_rank_correlations
from pseudospace.coordinate_evaluation import interpolate_curves

P1_REFS = {'mouse': ['mouse_sn_male', 'mouse_micro'], 'human': ['human_sn_halfB'],
           'control': ['mouse_sn_male', 'mouse_micro'], 'IR': ['mouse_sn_male', 'mouse_micro']}
PRIMARY_P1 = {'mouse_sn_male', 'human_sn_halfB'}
P1W_EVAL = {'S1': 'mouse_sn_male_S2mS1', 'S2': 'mouse_sn_male_S3mS1', 'S3': 'mouse_sn_male_S3mS2'}
P1W_MICRO = {'S1': 'mouse_micro_S2mS1', 'S2': 'mouse_micro_S3mS1', 'S3': 'mouse_micro_S3mS2'}


def context(ds, rows=None, segment=None, label='all PT units'):
    d = D[ds]
    rows = np.arange(len(d['ids'])) if rows is None else np.asarray(rows)
    c = {'ds': ds, 'label': label, 'rows': rows, 'groups': d['groups'], 'ref': d['ref'], 'case_label': d['case_label'],
         'genes': d['eval_genes'], 'fold': d['eval_fold'], 'EX': d['EX'][rows]}
    for key in ('specimen', 'group', 'segment', 'log_library', 'n_spots', 'n_detected'):
        c[key] = d[key][rows]
    if segment is not None:
        c['segment'] = np.asarray(segment).astype(str)
    short = pd.Series(c['segment']).str.replace('PT-', '', regex=False).to_numpy()
    c['seg_group'] = np.where(c['group'] == 'human', np.where(short == 'S3', 'S3', 'early'), short)
    c['within'] = np.char.add(np.char.add(c['specimen'].astype(str), '|'), c['seg_group'].astype(str))
    return c


def species_r(c, X, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(X[keep], z[keep], c['specimen'][keep])
    table.index = genes
    return pd.DataFrame({g: table.reindex(columns=names).mean(axis=1, skipna=False) for g, names in c['groups'].items()})


def within_r(c, X, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(X[keep], z[keep], c['within'][keep])
    table.index = genes
    return table


def protected(c, folds, function):
    """Fold-k evaluation genes scored on the fold-k refit (notebook 35's fold protection)."""
    parts = []
    for k in range(N_FOLDS):
        cols = c['fold'] == k
        parts.append(function(c, c['EX'][:, cols], folds[k], c['genes'][cols]))
    return pd.concat(parts).reindex(c['genes'])


def p1_rows(c, R, arm):
    rows = []
    for g in c['groups']:
        for ref in P1_REFS[g]:
            mask = evaluation_mask(c['ds'], g, ref).reindex(c['genes']).to_numpy(bool)
            rho, n = concordance(R.loc[mask, g], contrast[ref].reindex(c['genes'][mask]))
            rows.append({'arm': arm, 'context': c['label'], 'group': g, 'reference': ref,
                         'primary': ref in PRIMARY_P1, 'rho': rho, 'genes': n})
    return rows


def p1w_rows(c, RW, arm):
    rows = []
    for g, names in c['groups'].items():
        if g == 'human':
            cells = [('early', 'human_sn_halfB', True), ('S3', 'human_sn_halfB', True)]
        else:
            cells = [(s, P1W_EVAL[s], True) for s in P1W_EVAL] + [(s, P1W_MICRO[s], False) for s in P1W_MICRO]
        for seg, ref, primary in cells:
            mask = evaluation_mask(c['ds'], g, ref).reindex(c['genes']).to_numpy(bool)
            for n in names:
                column = f'{n}|{seg}'
                values = RW[column] if column in RW else pd.Series(np.nan, index=RW.index)
                rho, count = concordance(values[mask], contrast[ref].reindex(c['genes'][mask]))
                rows.append({'arm': arm, 'context': c['label'], 'group': g, 'segment': seg, 'specimen': n,
                             'reference': ref, 'primary': primary, 'rho': rho, 'genes': count})
    return rows


def cells(metric, arm, c, table, **extra):
    group_of = {n: g for g, names in c['groups'].items() for n in names}
    return [{'metric': metric, 'arm': arm, 'context': c['label'], 'group': group_of[r.specimen], 'specimen': r.specimen,
             'segment': r.segment, 'rho': r.rho, **extra} for r in table.itertuples()]


def gene_fold_rows(c, arm, coords):
    rows = []
    for a, b in ((0, 1), (0, 2), (1, 2)):
        rows += cells('gene-fold agreement', arm, c, within_segment_agreement(
            coords[f'fold{a}'], coords[f'fold{b}'], c['specimen'], c['segment']), pair=f'{a}-{b}')
    return rows


def scaled(c, z):
    z = np.asarray(z, float)
    if not np.isfinite(z).any():
        return z
    return scale71(z, c['specimen'], c['segment'])


def gap_rows(c, arm, coords, keys):
    rows = []
    for key in keys:
        z = coords.get(key)
        if z is None:
            continue
        if not np.isfinite(z).any():
            rows.append({'arm': arm, 'context': c['label'], 'refit': key, 'segment': 'all', 'gap': np.nan})
            continue
        gap, _ = registration_gap(scaled(c, z), c['group'], c['specimen'], c['segment'], first=c['case_label'],
                                  second=c['ref'])
        rows += [{'arm': arm, 'context': c['label'], 'refit': key, 'segment': seg, 'gap': float(v)} for seg, v in gap.items()]
    return rows


def noise_rows(c, arm, coords):
    """Notebook 60's noise side: fold and count-split noise SD in segment units and relative to the spread."""
    z = scaled(c, coords['full'])
    keep = np.isfinite(z) & (z >= 0) & (z <= 1)
    trans = logistic_transitions(z[keep], c['segment'][keep], c['specimen'][keep]).pivot(
        index='specimen', columns='transition', values='position')
    lo = max(z[keep & (c['specimen'] == n)].min() for n in np.unique(c['specimen']))
    hi = min(z[keep & (c['specimen'] == n)].max() for n in np.unique(c['specimen']))
    variances = {}
    for name, keys in (('fold', ['fold0', 'fold1', 'fold2']), ('count split', ['half_A', 'half_B'])):
        found = {k: scaled(c, coords[k]) for k in keys if k in coords and np.isfinite(coords[k]).any()}
        if len(found) >= 2:
            variances[name] = fold_noise_variance(found, c['specimen'], c['segment'], c['group'])[1].mean(axis=1)
    rows = []
    for n in np.unique(c['specimen']):
        t12, t23 = trans.loc[n, 'S1→S2'], trans.loc[n, 'S2→S3']
        lengths = {'PT-S1': t12 - lo, 'PT-S2': t23 - t12, 'PT-S3': hi - t23}
        g = c['group'][c['specimen'] == n][0]
        for seg, length in lengths.items():
            m = keep & (c['specimen'] == n) & (c['segment'] == seg)
            spread = float(np.std(z[m]))
            row = {'arm': arm, 'context': c['label'], 'group': g, 'specimen': n, 'segment': seg,
                   'segment length': length, 'position SD within segment': spread}
            for name, var in variances.items():
                if (g, n, seg) in var.index:
                    sd = float(np.sqrt(var.loc[(g, n, seg)]))
                    row[f'{name} noise SD (segment units)'] = sd / length if length > 0 else np.nan
                    row[f'{name} noise ÷ spread'] = sd / spread if spread > 0 else np.nan
            rows.append(row)
    return rows


def summary_of(c, arm, P1, P1W, CELLS, GAPS):
    """One arm's summary values in one context (notebook 51's definitions)."""
    out = {}
    for g in c['groups']:
        ref = 'human_sn_halfB' if g == 'human' else 'mouse_sn_male'
        hit = P1[(P1.arm == arm) & (P1.context == c['label']) & (P1.group == g) & (P1.reference == ref)]
        out[f'P1 {g}'] = float(hit.rho.iloc[0]) if len(hit) else np.nan
        w = P1W[(P1W.arm == arm) & (P1W.context == c['label']) & (P1W.group == g) & P1W.primary]
        out[f'P1w {g}'] = float(w.rho.mean()) if len(w) else np.nan
        f = CELLS[(CELLS.metric == 'gene-fold agreement') & (CELLS.arm == arm) & (CELLS.context == c['label'])
                  & (CELLS.group == g)]
        out[f'gene-fold {g}'] = float(f.groupby('segment').rho.median().mean()) if len(f) else np.nan
        out[f'gene-fold {g} S1'] = float(f[f.segment.eq('PT-S1')].rho.median()) if len(f) else np.nan
    for metric, key in (('P5 equal depth', 'P5'), ('LOSO', 'LOSO'), ('count split A vs B', 'count split A vs B'),
                        ('numerical floor', 'floor'), ('12 threads vs default', '12 threads')):
        t = CELLS[(CELLS.metric == metric) & (CELLS.arm == arm) & (CELLS.context == c['label'])]
        out[key] = float(t.rho.median()) if len(t) else np.nan
        if metric == 'P5 equal depth':
            out['P5 human S1'] = float(t[t.group.eq('human') & t.segment.eq('PT-S1')].rho.mean()) if len(t) else np.nan
    gaps = GAPS[(GAPS.arm == arm) & (GAPS.context == c['label']) & GAPS.refit.isin(['fold0', 'fold1', 'fold2', 'half_A', 'half_B'])]
    out['registration gap'] = float(gaps.gap.abs().max()) if gaps.gap.notna().any() else np.nan
    full = GAPS[(GAPS.arm == arm) & (GAPS.context == c['label']) & GAPS.refit.eq('full')]
    out['gap of the full fit'] = float(full.gap.abs().max()) if full.gap.notna().any() else np.nan
    return out


def evaluate(c, arm, coords, P1_ROWS, P1W_ROWS, CELL_ROWS, GAP_ROWS, NOISE_ROWS):
    """Every coordinate metric of one arm in one context; the rows are appended to the given lists."""
    folds = [coords.get(f'fold{k}') for k in range(N_FOLDS)]
    if all(f is not None for f in folds):
        R = protected(c, folds, species_r)
        RW = protected(c, folds, within_r)
        P1_ROWS += p1_rows(c, R, arm)
        P1W_ROWS += p1w_rows(c, RW, arm)
        CELL_ROWS += gene_fold_rows(c, arm, coords)
    full = coords['full']
    for metric, key in (('P5 equal depth', 'ed'), ('numerical floor', 'floor'), ('identity', 'identity'),
                        ('12 threads vs default', 'identity_12threads'), ('LOSO', 'loso')):
        if key in coords:
            CELL_ROWS += cells(metric, arm, c, within_segment_agreement(coords[key], full, c['specimen'], c['segment']))
    if 'half_A' in coords and 'half_B' in coords:
        CELL_ROWS += cells('count split A vs B', arm, c, within_segment_agreement(coords['half_A'], coords['half_B'],
                                                                                  c['specimen'], c['segment']))
    CELL_ROWS += cells('position vs log library (before)', arm, c,
                       within_segment_agreement(full, c['log_library'], c['specimen'], c['segment']))
    if 'ed' in coords:
        CELL_ROWS += cells('position vs log library (after equal depth)', arm, c,
                           within_segment_agreement(coords['ed'], c['log_library'], c['specimen'], c['segment']))
    if 'injection' in coords:
        CELL_ROWS += cells('injection refit vs full', arm, c, within_segment_agreement(coords['injection'], full,
                                                                                       c['specimen'], c['segment']))
    GAP_ROWS += gap_rows(c, arm, coords, ['full', 'fold0', 'fold1', 'fold2', 'half_A', 'half_B'])
    if np.isfinite(full).sum() > 100:
        NOISE_ROWS += noise_rows(c, arm, coords)

## 7 · LOSO, curve only (notebook 36's design, through the package's public curve functions)

For each specimen n: fit `curve_pseudospace` on the base embedding without n's units (rooted at the base root, or at
`choose_root` on the remaining units when the root is in n), place n's units with `project_onto_curve` and
`curve_pseudotime`, signed by the training orientation, and compare with the base positions within n's segments.
Harmony saw every specimen, as in notebook 36, so this tests the curve, not the integration.

In [ ]:
from pseudospace_reconstruction import trajectory as ptraj


def loso_payload(ds):
    p2 = ad.read_h5ad(RUNS[ds] / '03_integration' / 'pass2.h5ad', backed='r')
    X = np.asarray(p2.obsm['X_harmony'], float)
    obs = p2.obs[['sample', 'leiden_coarse', 'broad_tubule_marker_call']].copy()
    names = pd.Index(p2.obs_names.astype(str))
    p2.file.close()
    table = pd.read_parquet(RUNS[ds] / '04_pseudospace' / 'pseudospace.parquet').reindex(names)
    axis = table.marker_axis.to_numpy(float)
    arguments = pipeline.stage04_arguments(DATASET_CONFIG[ds])
    params = arguments['params']
    root_name = str(np.load(RUNS[ds] / '04_pseudospace' / 'curve_nodes.npz')['root_obs_name'])
    out = np.full(len(names), np.nan)
    roots = {}
    for n in sorted(obs['sample'].astype(str).unique()):
        train = obs['sample'].astype(str).to_numpy() != n
        train_names = names[train]
        if root_name in set(train_names):
            root = int(train_names.get_loc(root_name))
        else:
            sub = ad.AnnData(obs=obs[train].copy())
            sub.obsm['X_harmony'] = X[train]
            root, _ = ptraj.choose_root(sub, axis[train], start_label=arguments['start_family'], rep_key='X_harmony')
        fit = ptraj.curve_pseudospace(X[train], root, axis[train], params)
        held = X[~train][:, :params.n_dims] if params.n_dims else X[~train]
        projection = ptraj.project_onto_curve(held, fit.fit.curve)
        out[~train] = fit.sign * ptraj.curve_pseudotime(fit.fit.curve, projection, fit.fit.root_node)
        roots[n] = str(train_names[root])
    return {'unit': np.asarray(names, dtype=str), 'loso': out, 'roots': np.array(json.dumps(roots))}


for ds, d in D.items():
    p = cached_payload(f'loso_{ds}', lambda ds=ds: loso_payload(ds), root=cache, params={'design': 'curve only'},
                       inputs={**BASE_KEY[ds], 'pass2': RUNS[ds] / '03_integration' / 'pass2.h5ad'},
                       code=digest([NOTEBOOK_LOGIC_VERSION]))
    COORD[ds]['package']['loso'] = refit_module.refit_position(p, d['ids'], 'loso')
    print(ds, 'LOSO roots:', json.loads(str(p['roots'])))
READING.append('LOSO for the package DPT is not computed: the curve-only design has no DPT analogue (notebook 36 '
               'reported LOSO for scFates arms only).')

## 8 · Coordinate metrics on all PT units (package and package DPT)

In [ ]:
P1_ROWS, P1W_ROWS, CELL_ROWS, GAP_ROWS, NOISE_ROWS = [], [], [], [], []
CTX = {ds: context(ds, label=f'{ds}: all PT units') for ds in DATASETS}
for ds in DATASETS:
    c = CTX[ds]
    for arm in ('package', 'package DPT'):
        evaluate(c, arm, COORD[ds][arm], P1_ROWS, P1W_ROWS, CELL_ROWS, GAP_ROWS, NOISE_ROWS)
    labels_z = pd.Series(c['segment']).map({'PT-S1': 0., 'PT-S2': 1., 'PT-S3': 2.}).to_numpy()
    P1_ROWS += p1_rows(c, species_r(c, c['EX'], labels_z, c['genes']), 'segment labels')
    P1W_ROWS += p1w_rows(c, within_r(c, c['EX'], c['log_library'], c['genes']), 'read depth (nuisance reference)')
P1 = pd.DataFrame(P1_ROWS)
P1W = pd.DataFrame(P1W_ROWS)
CELLS = pd.DataFrame(CELL_ROWS)
GAPS = pd.DataFrame(GAP_ROWS)
NOISE = pd.DataFrame(NOISE_ROWS)
SUMMARY = {ds: pd.DataFrame({arm: summary_of(CTX[ds], arm, P1, P1W, CELLS, GAPS)
                             for arm in ('package', 'package DPT', 'segment labels')}) for ds in DATASETS}
for ds in DATASETS:
    display(SUMMARY[ds].round(3).rename_axis(f'{ds} · all PT units'))
display(P1.pivot_table(index=['context', 'arm'], columns=['group', 'reference'], values='rho').round(3))

## 9 · P2 · Does the coordinate absorb an injected group-by-position effect?

In the case group's PT units (human; IR in AKI), 10% of the base pass-2 integration features were multiplied by
exp(±log 2 · (2s − 1)) along the base position. The interaction r_case − r_reference on the injected genes is
computed on the original data, on the injected data with the base coordinate (frozen) and with the refitted one.
Material absorption (notebook 35's flag): median ratio < 0.85, or case within-segment ρ(refit, base) < 0.90.

In [ ]:
P2_ROWS = []
for ds, d in D.items():
    c = CTX[ds]
    b = BASE[ds]
    counts = sparse.csr_matrix(b['units'].layers['counts']).astype(np.int64)
    rows = b['pt_rows'][d['case']]
    f = INJECTION_LOG_AMPLITUDE * (2 * b['s0'][d['case']] - 1)
    cols = b['units'].var_names.get_indexer(b['injected'])
    injected_counts = scale_counts(counts, rows, cols, f[:, None] * b['signs'][None, :],
                                   np.random.default_rng(INJECTION_SEED))[b['pt_rows']]
    del counts
    inj_cols = d['genes'].get_indexer(b['injected'])
    injected_eval = log_normalize(injected_counts, np.asarray(injected_counts.sum(axis=1)).ravel().astype(float))
    genes = pd.Index(b['injected'])

    def interaction(X, z):
        r = species_r(c, X, z, genes)
        return (r[d['case_label']] - r[d['ref']]).to_numpy()

    for arm in ('package', 'package DPT'):
        z0, z1 = COORD[ds][arm]['full'], COORD[ds][arm]['injection']
        original = interaction(d['Y'][:, inj_cols], z0)
        frozen = interaction(injected_eval[:, inj_cols], z0)
        refitted = interaction(injected_eval[:, inj_cols], z1) if np.isfinite(z1).any() else np.full(len(genes), np.nan)
        moved = within_segment_agreement(z1, z0, c['specimen'], c['segment'])
        case_cells = moved[moved.specimen.isin(GROUPS[ds][d['case_label']])]
        rec = {'dataset': ds, 'arm': arm, **absorption_ratio(original, frozen, refitted, floor=ABSORPTION_FLOOR),
               'case within-segment rho': float(case_cells.rho.median()),
               'frozen change in the injected direction': float(np.mean(np.sign(frozen - original) == b['signs']))}
        rec['material absorption (notebook 35 flag)'] = bool(not (rec['median_ratio'] >= .85)
                                                             or not (rec['case within-segment rho'] >= .90))
        P2_ROWS.append(rec)
P2 = pd.DataFrame(P2_ROWS)
P2.to_csv(tables / 'p2_absorption.csv', index=False)
display(P2.round(3))

## 10 · C1 · Canonical order: transitions, composition and the late S3 markers

Per specimen, the S1→S2 and S2→S3 transitions are where a logistic regression of the later label on position
(notebook 37) gives 0.5, on notebook 71's scale over the common support. **C1 (new rule):** both transitions lie
inside the support and in order in every specimen, and Slc7a13 and Slc22a7 rise along the coordinate (Spearman > 0)
in both mice. Both are PT-S3 panel genes (they set the labels, not the curve), so this is a sanity check.

In [ ]:
TRANSITIONS, COMPOSITION, LATE = [], [], []
for ds, d in D.items():
    keep = d['support']
    t = logistic_transitions(d['pos71'][keep], d['segment'][keep], d['specimen'][keep])
    t['dataset'] = ds
    t['group'] = t.specimen.map({n: g for g, names in d['groups'].items() for n in names})
    TRANSITIONS.append(t)
    for n in np.unique(d['specimen']):
        m = keep & (d['specimen'] == n)
        bins = pd.cut(d['pos71'][m], np.linspace(0, 1, 21), include_lowest=True, labels=False)
        share = pd.crosstab(bins, d['segment'][m], normalize='index').reindex(columns=SEGMENTS, fill_value=0)
        COMPOSITION.append(share.assign(dataset=ds, specimen=n, bin_mid=(share.index + .5) / 20).reset_index(drop=True))
        for gene in ('Slc7a13', 'Slc22a7'):
            col = d['genes'].get_loc(gene)
            y = d['Y'][d['specimen'] == n][:, col].toarray().ravel()
            LATE.append({'dataset': ds, 'specimen': n, 'gene': gene,
                         'Spearman with position': spearmanr(y, d['raw'][d['specimen'] == n]).statistic})
TRANSITIONS = pd.concat(TRANSITIONS, ignore_index=True)
COMPOSITION = pd.concat(COMPOSITION, ignore_index=True)
LATE = pd.DataFrame(LATE)
TRANSITIONS.to_csv(tables / 'transitions.csv', index=False)
COMPOSITION.to_csv(tables / 'composition.csv', index=False)
LATE.to_csv(tables / 'late_markers.csv', index=False)
wide = TRANSITIONS.pivot_table(index=['dataset', 'group', 'specimen'], columns='transition', values='position')
wide['ordered inside the support'] = (wide['S1→S2'] > 0) & (wide['S1→S2'] < wide['S2→S3']) & (wide['S2→S3'] < 1)
display(wide.round(3))
display(LATE.pivot_table(index=['dataset', 'specimen'], columns='gene', values='Spearman with position').round(3))
SHIFTS = (TRANSITIONS.groupby(['dataset', 'group', 'transition']).position.mean().unstack('group'))
display(SHIFTS.round(3).rename_axis(['dataset', 'transition (group means)']))

## 11 · Canonical markers along the coordinate (Figure 2a, descriptive)

NB group and specimen curves from notebook 71's `standard_gene_model` (raw counts, offset log of the mt-free
library, cubic B-spline with 6 df, notebook 71's position and grid) for Slc5a2, Slc5a12, Slc34a1, Slc22a6, Slc7a13
and Slc22a7, with specimen means in 20 equal-count bins. Only Slc34a1 is outside the package's panels and anchors.

In [ ]:
from pseudospace.package_inputs import aki_inputs, hvm_inputs
from pseudospace.pathway_pipelines import camera_pr, moderated_f, specimen_shape_lr, standard_gene_model

MARKERS = ['Slc5a2', 'Slc5a12', 'Slc34a1', 'Slc22a6', 'Slc7a13', 'Slc22a7']
orthologs = pd.read_csv(results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
INPUTS = {'hvm': hvm_inputs(UNITS['hvm'], data_root, orthologs, exclude_genes=MT)}
INPUTS['aki'] = aki_inputs(UNITS['aki'], data_root, orthologs,
                           INPUTS['hvm']['genes'] + INPUTS['hvm']['excluded_universe'], exclude_genes=MT)
for ds, inp in INPUTS.items():
    rows = D[ds]['ids'].get_indexer(inp['obs'].index)
    assert (rows >= 0).all()
    np.testing.assert_allclose(inp['position'], D[ds]['pos71'][rows], atol=1e-12)
    inp['rows'] = rows
print({ds: f"{len(inp['genes']):,} universe genes, {len(inp['gene_sets']):,} gene sets, {len(inp['rows']):,} units"
       for ds, inp in INPUTS.items()})
# Notebook 71's own human vs mouse universe and gene sets (its tables keep notebook 69's dataset names): C13 runs on
# them, and A3 reads T_spatial from its cached condition-split gene model. They must equal the loader's.
universe71 = pd.read_csv(nb71 / 'tables' / 'universe_human_vs_mouse.csv')
U71 = universe71.loc[universe71.eligible.astype(bool), 'gene'].astype(str).tolist()
S71 = json.loads((nb71 / 'tables' / 'gene_sets_human_vs_mouse.json').read_text())
SAME71 = {'universe': U71 == list(INPUTS['hvm']['genes']), 'gene sets': S71 == INPUTS['hvm']['gene_sets']}
print("Notebook 71's universe and gene sets equal the loader's:", SAME71)
READING.append("Notebook 71's tables keep notebook 69's dataset names: hvm = 'human_vs_mouse', aki = 'aki_vs_control' "
               '(gene_statistics_human_vs_mouse.csv, universe_{ds}.csv, gene_sets_{ds}.json, method_table.csv, stage '
               f'gene_model_human_vs_mouse_species). Its universe and gene sets equal the loader\'s: {SAME71}.')


def gene_model_payload(model):
    out = {'grid': model.grid, 'reference': model.reference, 'case': model.case, 'delta': model.delta,
           **{f'stat|{c}': model.statistics[c].to_numpy(float) for c in ('T_spatial', 'alpha')},
           **{f'flag|{c}': model.statistics[c].to_numpy(bool) for c in ('converged', 'separated')}}
    for name, curve in model.specimen_curves.items():
        out[f'curve|{name}'] = curve
    return out


MARKER_CURVES = {}
for ds, inp in INPUTS.items():
    d = D[ds]
    cols = d['genes'].get_indexer(MARKERS)
    counts = d['counts'][inp['rows']][:, cols]
    MARKER_CURVES[ds] = cached_payload(
        f'markers_{ds}', lambda counts=counts, inp=inp: gene_model_payload(standard_gene_model(
            counts, inp['library'], inp['position'], inp['case'].astype(float), inp['specimen'], genes=MARKERS,
            basis_df=BASIS_DF, n_jobs=min(N_JOBS, 6))),
        root=cache, params={'genes': MARKERS, 'basis_df': BASIS_DF},
        inputs={'counts': counts, 'library': inp['library'], 'position': inp['position'], 'specimen': list(inp['specimen'])},
        code=digest([NOTEBOOK_LOGIC_VERSION]))
BIN_MEANS = []
for ds, inp in INPUTS.items():
    d = D[ds]
    counts = d['counts'][inp['rows']][:, d['genes'].get_indexer(MARKERS)].toarray()
    for n in np.unique(inp['specimen']):
        m = inp['specimen'] == n
        order = np.argsort(inp['position'][m])
        for chunk in np.array_split(order, 20):
            rows = np.flatnonzero(m)[chunk]
            rate = counts[rows].sum(axis=0) / inp['library'][rows].sum() * 1e4
            BIN_MEANS += [{'dataset': ds, 'specimen': n, 'position': float(inp['position'][rows].mean()), 'gene': g,
                           'log CP10k': float(np.log(r)) if r > 0 else np.nan} for g, r in zip(MARKERS, rate)]
BIN_MEANS = pd.DataFrame(BIN_MEANS)
BIN_MEANS.to_csv(tables / 'marker_bin_means.csv', index=False)
pd.concat([pd.DataFrame({'dataset': ds, 'gene': np.repeat(MARKERS, len(p['grid'])), 'position': np.tile(p['grid'], len(MARKERS)),
                         'reference log CP10k': (p['reference'] + np.log(1e4)).ravel(),
                         'case log CP10k': (p['case'] + np.log(1e4)).ravel()})
           for ds, p in MARKER_CURVES.items()]).to_csv(tables / 'marker_curves.csv', index=False)

## 12 · C11 positive controls and C12 attenuation (human vs mouse)

- **C11.** For genes whose external zonation differs between species (opposite sign, or mouse-zonated and
  human-flat), the coordinate interaction r_human − r_mouse (fold-protected) should carry the sign of the external
  interaction (human half B minus male mouse). Notebook 35's label contrast (S3 − early) is the reference.
- **C12.** For conserved held-out genes H (strict anchors that never built the coordinate), NB amplitudes (curve at
  the S3 median minus the S1 median, signed by the conserved direction) in each species. Flag (notebook 35):
  ≥ 70% of H genes with a smaller human amplitude, or a median notebook-71 T_spatial percentile ≥ 0.75.

In [ ]:
c = CTX['hvm']
pkg_folds = [COORD['hvm']['package'][f'fold{k}'] for k in range(N_FOLDS)]


def protected_subset(c, folds, genes):
    sub = dict(c)
    keep = pd.Index(c['genes']).get_indexer(genes)
    sub.update({'genes': c['genes'][keep], 'fold': c['fold'][keep], 'EX': c['EX'][:, keep]})
    return protected(sub, folds, species_r)


def label_contrast(c, genes):
    keep = pd.Index(c['genes']).get_indexer(genes)
    X = c['EX'][:, keep]
    out = {}
    for g, names in c['groups'].items():
        per = []
        for n in names:
            late = np.asarray(X[(c['specimen'] == n) & (c['segment'] == 'PT-S3')].mean(axis=0)).ravel()
            early = np.asarray(X[(c['specimen'] == n) & (c['segment'] != 'PT-S3')].mean(axis=0)).ravel()
            per.append(late - early)
        out[g] = np.mean(per, axis=0)
    return pd.DataFrame(out, index=c['genes'][keep])


external_interaction = contrast['human_sn_halfB'] - contrast['mouse_sn_male']
control_genes = [g for g in h['genes'][(D_OPPOSITE | D_FLAT).to_numpy()] if g in set(c['genes'])]
R_controls = {'package': protected_subset(c, pkg_folds, control_genes),
              'package DPT': protected_subset(c, [COORD['hvm']['package DPT'][f'fold{k}'] for k in range(N_FOLDS)],
                                              control_genes),
              'segment labels': label_contrast(c, control_genes)}
CONTROL_ROWS = []
for subset, mask in (('opposite sign', D_OPPOSITE), ('mouse-zonated, human-flat', D_FLAT),
                     ('all positive controls', D_OPPOSITE | D_FLAT)):
    idx = [g for g in h['genes'][mask.to_numpy()] if g in set(control_genes)]
    for arm, table in R_controls.items():
        ours = table.loc[idx, 'human'] - table.loc[idx, 'mouse']
        ok = ours.notna()
        CONTROL_ROWS.append({'set': subset, 'arm': arm, 'genes': int(ok.sum()),
                             'sign agreement': float((np.sign(ours[ok]) == np.sign(external_interaction[idx][ok])).mean()),
                             'Spearman': concordance(ours, external_interaction[idx])[0]})
CONTROLS = pd.DataFrame(CONTROL_ROWS)
CONTROLS.to_csv(tables / 'positive_controls.csv', index=False)
display(CONTROLS.pivot_table(index='arm', columns='set', values=['sign agreement', 'genes']).round(3))

inp = INPUTS['hvm']
h_genes = [g for g in h['genes'][H_SET.to_numpy()] if g in set(inp['genes'])]
h_cols = pd.Index(inp['genes']).get_indexer(h_genes)
a3_counts = inp['counts'][:, h_cols]
A3_MODEL = cached_payload(
    'a3_hvm', lambda: gene_model_payload(standard_gene_model(
        a3_counts, inp['library'], inp['position'], inp['case'].astype(float), inp['specimen'], genes=h_genes,
        basis_df=BASIS_DF, n_jobs=min(N_JOBS, 12))),
    root=cache, params={'genes': h_genes, 'basis_df': BASIS_DF},
    inputs={'counts': a3_counts, 'library': inp['library'], 'position': inp['position'], 'specimen': list(inp['specimen'])},
    code=digest([NOTEBOOK_LOGIC_VERSION]))
seg_in, spec_in = inp['segment'], inp['specimen']
s_S1 = np.mean([np.median(inp['position'][(spec_in == n) & (seg_in == 'PT-S1')]) for n in np.unique(spec_in)])
s_S3 = np.mean([np.median(inp['position'][(spec_in == n) & (seg_in == 'PT-S3')]) for n in np.unique(spec_in)])
sign_h = anchor_sign.reindex(h_genes).to_numpy()
amp = {g: (interpolate_curves(A3_MODEL[key], A3_MODEL['grid'], [s_S3])[:, 0]
           - interpolate_curves(A3_MODEL[key], A3_MODEL['grid'], [s_S1])[:, 0]) * sign_h
       for g, key in (('mouse', 'reference'), ('human', 'case'))}
lab_amp = {}
for g, names in GROUPS['hvm'].items():
    per = []
    for n in names:
        rate = {}
        for s in ('PT-S1', 'PT-S3'):
            m = (spec_in == n) & (seg_in == s)
            rate[s] = np.log((np.asarray(a3_counts[m].sum(axis=0)).ravel() + .5) / inp['library'][m].sum())
        per.append(rate['PT-S3'] - rate['PT-S1'])
    lab_amp[g] = np.mean(per, axis=0) * sign_h
ATTENUATION = pd.DataFrame({'gene': h_genes, 'amp_mouse': amp['mouse'], 'amp_human': amp['human'],
                            'label_amp_mouse': lab_amp['mouse'], 'label_amp_human': lab_amp['human'],
                            'external_human_smaller': (contrast['human_sn_halfB'].abs()
                                                       < contrast['mouse_sn_male'].abs()).reindex(h_genes).to_numpy()})


def nb71_payload(stage_glob):
    """Notebook 71's newest cached payload of one stage (read only), or None when it is not there yet."""
    files = sorted((nb71 / 'stage_cache').glob(stage_glob),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc']) \
        if (nb71 / 'stage_cache').is_dir() else []
    if not files:
        return None
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


T71 = nb71_payload('gene_model_human_vs_mouse_species__*.npz')
if T71 is not None and len(T71['stat|T_spatial']) == len(U71):
    percentile = pd.Series(T71['stat|T_spatial'], index=U71).rank(pct=True)
    ATTENUATION['T_spatial_percentile_71'] = percentile.reindex(h_genes).to_numpy()
else:
    ATTENUATION['T_spatial_percentile_71'] = np.nan
ATTENUATION.to_csv(tables / 'a3_attenuation.csv', index=False)
positive = ATTENUATION.amp_mouse.gt(0) & ATTENUATION.label_amp_mouse.gt(0)
A3 = pd.Series({
    'conserved held-out genes in the universe': len(h_genes),
    'fraction human amplitude < mouse (coordinate, NB)': float((ATTENUATION.amp_human < ATTENUATION.amp_mouse).mean()),
    'fraction human amplitude < mouse (segment labels)': float((ATTENUATION.label_amp_human < ATTENUATION.label_amp_mouse).mean()),
    'fraction |Δ human| < |Δ mouse| (external snRNA)': float(ATTENUATION.external_human_smaller.mean()),
    'median human/mouse amplitude ratio (coordinate)': float(np.median(ATTENUATION.amp_human[positive] / ATTENUATION.amp_mouse[positive])),
    'median human/mouse amplitude ratio (labels)': float(np.median(ATTENUATION.label_amp_human[positive] / ATTENUATION.label_amp_mouse[positive])),
    'median notebook-71 T_spatial percentile': float(ATTENUATION.T_spatial_percentile_71.median()),
    'S1 / S3 median positions': f'{s_S1:.3f} / {s_S3:.3f}'})
display(A3.to_frame('value'))
READING.append('A3 label amplitudes use specimen pseudobulk log rates (S3 − S1, counts over the mt-free library), the '
               'count analogue of notebook 35\'s log1p means, to match the NB curve amplitudes (decision D2).')

## 13 · C13 · Count splitting: does building and testing on the same reads inflate the positional test?

Half B of the count split (seed 35) is tested with notebook 71's positional F-test (condition split: NB gene
model, specimen-shape denominator, `squeezeVar` with the log mean count, z = Φ⁻¹(1 − p)) and cameraPR (0.01; call =
Up, FDR ≤ 0.05), on notebook 71's universe and gene sets, along three coordinates:
(a) the package refitted on half A (independent of B's noise), (b) the base coordinate, (c) the package refitted on
half B (circular). Each arm keeps its own common support; the offset is the mt-free half-B library.
**C13 (new rule):** not met if arm a calls fewer than half of arm b's calls.

In [ ]:
universe = pd.Index(U71)
mt_free = ~D['hvm']['genes'].isin(MT)
b = BASE['hvm']
_, HALF_B = poisson_count_split(sparse.csr_matrix(b['units'].layers['counts']).astype(np.int64), SPLIT_EPSILON,
                                np.random.default_rng(SPLIT_SEED))
HALF_B = HALF_B[b['pt_rows']]
G6_ARMS = {'a · refit on half A (independent)': COORD['hvm']['package']['half_A'],
           'b · base coordinate': COORD['hvm']['package']['full'],
           'c · refit on half B (circular)': COORD['hvm']['package']['half_B']}


def g6_arm(raw):
    d = D['hvm']
    finite = np.flatnonzero(np.isfinite(raw))
    mask, position = support_positions(raw[finite], d['specimen'][finite], d['segment'][finite])
    units = finite[mask]
    counts = HALF_B[units][:, d['genes'].get_indexer(universe)]
    library = np.asarray(HALF_B[units][:, mt_free].sum(axis=1)).ravel().astype(float)
    group = d['case'][units].astype(float)
    specimen = d['specimen'][units]
    model = standard_gene_model(counts, library, position, group, specimen, genes=list(universe), basis_df=BASIS_DF,
                                n_jobs=N_JOBS)
    den = specimen_shape_lr(counts, library, position, group, specimen, model.dispersion, basis_df=BASIS_DF,
                            n_jobs=N_JOBS)
    log_mean = np.log(np.maximum(np.asarray(counts.mean(axis=0)).ravel(), 1e-12))
    frame = moderated_f(model.statistics.T_spatial, DF1, np.asarray(den['LR_spec'], float), int(den['df2']),
                        covariate=log_mean, robust=False)
    test = camera_pr(frame.z, S71, inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
    return {'units': len(units), 'T_spatial': model.statistics.T_spatial.to_numpy(float), 'F': frame.F.to_numpy(float),
            'z': frame.z.to_numpy(float), 'p': frame.p.to_numpy(float), 'df2': int(den['df2']),
            'set_ids': np.asarray(test.pathway_id, str), 'set_fdr': test.fdr.to_numpy(float),
            'set_call': test.call.to_numpy(bool)}


G6 = {}
for arm, raw in G6_ARMS.items():
    G6[arm] = cached_payload(f'g6_{arm[0]}', lambda raw=raw: g6_arm(raw), root=cache,
                             params={'arm': arm, 'basis_df': BASIS_DF, 'df1': DF1, 'camera': CAMERA_COR},
                             inputs={'half_b': HALF_B, 'raw': raw, 'universe': list(universe),
                                     'sets': S71}, code=digest([NOTEBOOK_LOGIC_VERSION]))
calls = {arm: set(p['set_ids'][p['set_call'].astype(bool)]) for arm, p in G6.items()}
base_b = G6['b · base coordinate']
G6_ROWS = []
for arm, p in G6.items():
    G6_ROWS.append({'arm': arm, 'units': int(p['units']), 'median T_spatial': float(np.median(p['T_spatial'])),
                    'median F': float(np.median(p['F'])),
                    'median F ÷ arm a': float(np.median(p['F']) / np.median(G6['a · refit on half A (independent)']['F'])),
                    'genes p ≤ 0.05': int((p['p'] <= .05).sum()), 'pathway calls': len(calls[arm]),
                    'calls shared with arm b': len(calls[arm] & calls['b · base coordinate']),
                    'Spearman z vs arm b': spearmanr(p['z'], base_b['z']).statistic})
calls71 = None
tests71 = sorted(nb71.glob('tables/pathway_tests_human_vs_mouse_cameraPR_species.csv'))
if tests71:
    t71 = pd.read_csv(tests71[0])
    calls71 = set(t71.pathway_id[t71.call.astype(bool)])
    READING.append(f'G6 tested sets equal notebook 71\'s ({tests71[0].name}): {set(t71.pathway_id) == set(S71)}.')
    for row in G6_ROWS:
        row["notebook 71's calls retained"] = len(calls[row['arm']] & calls71)
G6_TABLE = pd.DataFrame(G6_ROWS)
G6_TABLE.to_csv(tables / 'g6_count_split.csv', index=False)
display(G6_TABLE.round(3))

## 14 · C15 · P4 and P4c: held-out genes across specimens (notebooks 36 and 51)

For each group, fold and direction (one specimen trains, the other is predicted), fold-k evaluation genes are
predicted from the fold-k refit: P4 against the training specimen's segment means; P4c adds a within-segment spline
of the coordinate (df 4) to the segment + coverage quadratic oracle (log library, log bins, log detected genes).
Units are those with a finite fold-k position in every arm of the context. **C15 (claim gate):** P4c > 0 in ≥ 5/6
mouse and ≥ 4/6 human cells. A sensitivity (not used by C15) repeats both on the units inside the common support:
a few PT units sit far along the nephron-wide curve, where the within-segment spline of P4c extrapolates.

In [ ]:
from pseudospace.coordinate_arms import heldout_prediction
from pseudospace.landmark_position import residual_spline_gain, scaled_gain
from pseudospace.within_segment_validation import fit_segment_covariate_oracle, predict_segment_covariate_oracle


def p4_rows(c, arms):
    """arms: {arm: [fold0, fold1, fold2] coordinates on the context's units}."""
    rows = []
    covariates = np.column_stack([c['log_library'], np.log(c['n_spots']), np.log(c['n_detected'])])
    anatomy = pd.Series(c['segment']).map({'PT-S1': 0, 'PT-S2': 1, 'PT-S3': 2}).to_numpy()
    for g, names in c['groups'].items():
        ref = 'human_sn_halfB' if g == 'human' else 'mouse_sn_male'
        genes_g = evaluation_mask(c['ds'], g, ref).reindex(c['genes']).to_numpy(bool)
        for k in range(N_FOLDS):
            cols = genes_g & (c['fold'] == k)
            # A failed refit (all NaN, notebook 36's rule) gives NaN rows and does not shrink the other arms' units.
            live = {arm: folds for arm, folds in arms.items() if np.isfinite(folds[k]).any()}
            for arm in set(arms) - set(live):
                rows += [{'arm': arm, 'context': c['label'], 'group': g, 'fold': k, 'train': t, 'units': 0,
                          'P4': np.nan, 'P4c': np.nan} for t in names]
            if not live:
                continue
            usable = np.all([np.isfinite(f[k]) for f in live.values()], axis=0)
            for train, test in ((names[0], names[1]), (names[1], names[0])):
                tr, te = usable & (c['specimen'] == train), usable & (c['specimen'] == test)
                ytr, yte = c['EX'][tr][:, cols].toarray(), c['EX'][te][:, cols].toarray()
                oracle = fit_segment_covariate_oracle(ytr, covariates[tr], anatomy[tr], c['specimen'][tr], degree=2)
                base_tr = predict_segment_covariate_oracle(oracle, covariates[tr], anatomy[tr])
                base_te = predict_segment_covariate_oracle(oracle, covariates[te], anatomy[te])
                for arm, folds in live.items():
                    z = folds[k]
                    p4 = heldout_prediction(ytr, z[tr], c['segment'][tr], yte, z[te], c['segment'][te], basis_df=BASIS_DF)
                    spline = residual_spline_gain(ytr, z[tr], c['segment'][tr], base_tr, yte, z[te], c['segment'][te],
                                                  base_te, basis_df=P4C_DF)
                    var = np.maximum(p4['train_var'], 1e-12)
                    rows.append({'arm': arm, 'context': c['label'], 'group': g, 'fold': k, 'train': train,
                                 'units': int(tr.sum() + te.sum()),
                                 'P4': float(1 - np.sum(p4['coordinate_mse'] / var) / np.sum(p4['segment_mse'] / var)),
                                 'P4c': scaled_gain(spline['base_mse'], spline['coordinate_mse'], spline['train_var'])})
    return rows


P4_ROWS = []
for ds in DATASETS:
    P4_ROWS += p4_rows(CTX[ds], {arm: [COORD[ds][arm][f'fold{k}'] for k in range(N_FOLDS)]
                                 for arm in ('package', 'package DPT')})
    # Sensitivity, not used by C15: the units inside the common support. A few PT units sit far along the
    # nephron-wide curve (beyond the PT stretch); the within-segment spline of P4c extrapolates at them.
    rows_s = np.flatnonzero(D[ds]['support'])
    P4_ROWS += p4_rows(context(ds, rows_s, label=f'{ds}: common-support units (sensitivity)'),
                       {arm: [COORD[ds][arm][f'fold{k}'][rows_s] for k in range(N_FOLDS)] for arm in ('package', 'package DPT')})
P4 = pd.DataFrame(P4_ROWS)
display(P4.groupby(['context', 'group', 'arm'])[['P4', 'P4c']].agg(['mean', lambda v: int((v > 0).sum())]).round(4))

## 15 · C14 · The noise side of the resolution budget (notebook 60)

Per specimen × segment: fold noise SD (Var(z_a − z_b)/2 over the gene-fold refits, averaged over pairs) and
count-split noise SD, in segment units (logistic transitions) and relative to the within-segment SD of positions,
on notebook 71's scale. **C14 (new reading rule):** a cell is noise-dominated when fold noise ÷ spread ≥ 1; not met
when ≥ 6 of the 12 human vs mouse cells are. Count-split noise comes from half-read refits and overstates
full-data noise; it is reported, not used in the rule.

In [ ]:
NOISE_SUMMARY = (NOISE.assign(noise_dominated=NOISE['fold noise ÷ spread'] >= 1)
                 .groupby(['context', 'arm', 'group', 'segment'])
                 [['fold noise SD (segment units)', 'fold noise ÷ spread', 'count split noise ÷ spread', 'noise_dominated']]
                 .agg({'fold noise SD (segment units)': 'mean', 'fold noise ÷ spread': 'mean',
                       'count split noise ÷ spread': 'mean', 'noise_dominated': 'sum'}))
display(NOISE_SUMMARY.round(3))

## 16 · Comparators on the matched set M (section 6 of the protocol; separable)

This section runs by default under D4. Comparators enter only through their cached positions on matched units (no
rebuild on the package's units, decision O2).
- **M (human vs mouse):** package PT units whose `old_structure_id` (notebook 70's mutual-best IoU ≥ 0.5 match) is
  an SCF13 PT structure. **M (AKI):** units whose old structure has a finite `pt_subset_scanpy_dpt`.
- Every arm on M uses the package counts and labels, gene set E, the references and the folds. A sensitivity
  repeats the within-segment metrics with the old reviewed labels (human vs mouse).
- **Recomputed on M:** P1/P1w (fold-protected), gene-fold, P5, A vs B, gap, LOSO (SCF13 only among comparators),
  injection within-segment ρ, noise budget and P4/P4c.
- **Quoted from the comparators' own units** (they need the old perturbed counts): the P2 ratio, the downstream
  gate, LRS-Z's P5, halves and gap, LCP's LOSO.
- Mouse comparators were built from the old polygons' counts, so on M they are scored against a matched polygon's
  expression; their own-unit values (notebook 53) are shown beside their M values as the matching penalty.

In [ ]:
import glob


def payload(folder, stage, key):
    files = glob.glob(str(folder / f'{stage}__*.npz'))
    if len(files) != 1:
        raise FileNotFoundError(f'{folder}: expected one {stage} payload, found {len(files)}')
    with np.load(files[0], allow_pickle=True) as stored:
        return np.asarray(stored[key], float)


saved = ad.read_h5ad(results_root / 'minimal_pt_scfates' / 'cross_species_pt_scfates.h5ad', backed='r')
saved_pt = saved.obs[['sample', 'segment_class', 'comparison_species', 'shared_pseudospace', 'dpt_pseudospace']].copy()
saved.file.close()
saved_ids = pd.Index(saved_pt.index.astype(str))
nb35c = results_root / 'pt_reconstruction_v2' / 'nb35' / 'stage_cache'
nb36c = results_root / 'pt_reconstruction_v2' / 'nb36' / 'stage_cache'
nb51c = results_root / 'pt_reconstruction_v3' / 'nb51' / 'stage_cache'
coords_dir = results_root / 'pt_reconstruction_v3' / 'coordinates'
SCF = saved_pt.shared_pseudospace.to_numpy(float)
assert spearmanr(payload(nb35c, 'refit_reproduction', 'scfates'), SCF).statistic > .999
OLD = {'SCF13': {'full': SCF, **{f'fold{k}': payload(nb35c, f'refit_fold{k}', 'scfates') for k in range(N_FOLDS)},
                 'ed': payload(nb35c, 'refit_equal_depth', 'scfates'), 'half_A': payload(nb35c, 'refit_split_A', 'scfates'),
                 'half_B': payload(nb35c, 'refit_split_B', 'scfates'), 'injection': payload(nb35c, 'refit_injection', 'scfates'),
                 'floor': payload(nb35c, 'refit_numerical_floor', 'scfates'), 'loso': payload(nb36c, 'loso_SCF13', 'loso')},
       'DPT13': {'full': saved_pt.dpt_pseudospace.to_numpy(float),
                 **{f'fold{k}': payload(nb35c, f'refit_fold{k}', 'dpt') for k in range(N_FOLDS)},
                 'ed': payload(nb35c, 'refit_equal_depth', 'dpt'), 'half_A': payload(nb35c, 'refit_split_A', 'dpt'),
                 'half_B': payload(nb35c, 'refit_split_B', 'dpt'), 'injection': payload(nb35c, 'refit_injection', 'dpt'),
                 'floor': payload(nb35c, 'refit_numerical_floor', 'dpt')}}
lcp_table = pd.read_csv(results_root / 'pt_reconstruction_v3' / 'lcp_coordinate.csv')
lcp_order = {sp: lcp_table.index[lcp_table.species.eq(sp)] for sp in ('mouse', 'human')}


def lcp(stage, species=('mouse', 'human')):
    values = pd.Series(lcp_table.position.to_numpy(float), index=lcp_table.structure_id.astype(str))
    for sp in species:
        values.iloc[lcp_order[sp]] = payload(nb51c, f'lcp_{sp}_{stage}', 'mean')
    return values.reindex(saved_ids).to_numpy(float)


for sp in ('mouse', 'human'):
    assert np.allclose(payload(nb51c, f'lcp_{sp}_full', 'mean'), lcp_table.position.to_numpy()[lcp_order[sp]])
OLD['LCP'] = {'full': lcp('full'), **{f'fold{k}': lcp(f'fold{k}') for k in range(N_FOLDS)}, 'ed': lcp('ed'),
              'half_A': lcp('half_A'), 'half_B': lcp('half_B'), 'injection': lcp('injection_own', species=('human',))}
lrsz = pd.read_csv(coords_dir / 'lrs_z.csv').set_index('structure_id').reindex(saved_ids)
OLD['LRS-Z'] = {'full': lrsz.position.to_numpy(float), **{f'fold{k}': lrsz[f'fold_{k}'].to_numpy(float) for k in range(N_FOLDS)}}

d = D['hvm']
old_ids = UNITS['hvm'].obs.old_structure_id.astype(object)
M_rows = np.flatnonzero(old_ids.isin(saved_ids).to_numpy())
M_index = saved_ids.get_indexer(old_ids.iloc[M_rows].astype(str))
old_labels = saved_pt.segment_class.astype(str).to_numpy()[M_index]
CTX_M = {'package labels': context('hvm', M_rows, label='M · package labels'),
         'old labels': context('hvm', M_rows, segment=old_labels, label='M · old reviewed labels')}
ARMS_M = {arm: {k: v[M_rows] for k, v in COORD['hvm'][arm].items()} for arm in ('package', 'package DPT')}
ARMS_M.update({arm: {k: v[M_index] for k, v in coords.items()} for arm, coords in OLD.items()})
M_P1, M_P1W, M_CELLS, M_GAPS, M_NOISE = [], [], [], [], []
for label, cm in CTX_M.items():
    for arm, coords in ARMS_M.items():
        evaluate(cm, arm, coords, M_P1, M_P1W, M_CELLS, M_GAPS, M_NOISE)
    labels_z = pd.Series(cm['segment']).map({'PT-S1': 0., 'PT-S2': 1., 'PT-S3': 2.}).to_numpy()
    M_P1 += p1_rows(cm, species_r(cm, cm['EX'], labels_z, cm['genes']), 'segment labels')
    M_P1W += p1w_rows(cm, within_r(cm, cm['EX'], cm['log_library'], cm['genes']), 'read depth (nuisance reference)')
M_P1, M_P1W, M_CELLS, M_GAPS, M_NOISE = (pd.DataFrame(x) for x in (M_P1, M_P1W, M_CELLS, M_GAPS, M_NOISE))
M_P4 = pd.DataFrame(p4_rows(CTX_M['package labels'], {arm: [c_[f'fold{k}'] for k in range(N_FOLDS)]
                                                       for arm, c_ in ARMS_M.items()}))
M_SUMMARY = {label: pd.DataFrame({arm: summary_of(cm, arm, M_P1, M_P1W, M_CELLS, M_GAPS)
                                  for arm in [*ARMS_M, 'segment labels']}) for label, cm in CTX_M.items()}
for label, table in M_SUMMARY.items():
    for arm in ARMS_M:
        sub = M_P4[(M_P4.arm == arm) & (M_P4.context == CTX_M[label]['label'])]
        for g in GROUPS['hvm']:
            table.loc[f'P4c > 0 cells {g}', arm] = float((sub[sub.group.eq(g)].P4c > 0).sum()) if len(sub) else np.nan
            table.loc[f'P4c mean {g}', arm] = float(sub[sub.group.eq(g)].P4c.mean()) if len(sub) else np.nan
            table.loc[f'P4 mean {g}', arm] = float(sub[sub.group.eq(g)].P4.mean()) if len(sub) else np.nan
        inj = M_CELLS[(M_CELLS.metric == 'injection refit vs full') & (M_CELLS.arm == arm)
                      & (M_CELLS.context == CTX_M[label]['label']) & M_CELLS.group.eq('human')]
        table.loc['injection: human within-segment ρ', arm] = float(inj.rho.median()) if len(inj) else np.nan
        noise = M_NOISE[(M_NOISE.arm == arm) & (M_NOISE.context == CTX_M[label]['label'])]
        table.loc['noise-dominated cells (of 12)', arm] = float((noise['fold noise ÷ spread'] >= 1).sum()) \
            if 'fold noise ÷ spread' in noise else np.nan
        table.loc['median fold noise ÷ spread', arm] = float(noise['fold noise ÷ spread'].median()) \
            if 'fold noise ÷ spread' in noise else np.nan
own = pd.read_csv(results_root / 'pt_reconstruction_v3' / 'nb53' / 'summary_by_arm.csv', index_col=0)
QUOTED = pd.DataFrame({
    'P2 median ratio (own units)': {'SCF13': 1.010, 'DPT13': 1.035, 'LCP': 1.028, 'LRS-Z': 1.004},
    'P2 source': {'SCF13': 'nb35 p2_absorption.csv', 'DPT13': 'nb35 p2_absorption.csv',
                  'LCP': 'nb51 p2_absorption.csv (own landmarks)', 'LRS-Z': 'nb53 p2_absorption.csv'},
    'downstream gate (own units, Gaussian joint test)': {'SCF13': '60 calls; relabeled 2, 0; NCDR 0.017 (nb53/ledger)',
                                                         'DPT13': '45 calls; relabeled 3, 0; NCDR 0.03 (SN1)',
                                                         'LCP': 'not run (abandoned before the gate, nb51)',
                                                         'LRS-Z': '46 calls; relabeled 0, 0; NCDR 0 (nb53/ledger)'},
    'LOSO (own units)': {'SCF13': np.nan, 'DPT13': np.nan, 'LCP': .99, 'LRS-Z': np.nan},
    **{f'{row} (own units)': own.loc[row].reindex(['SCF13', 'DPT13', 'LCP', 'LRS-Z']).to_dict()
       for row in ('P1 mouse', 'P1 human', 'P1w mouse', 'P1w human', 'gene-fold mouse', 'gene-fold human',
                   'human S1 gene-fold', 'P5 overall', 'P5 human S1', 'registration gap')}})
M_SUMMARY['package labels'].to_csv(tables / 'comparators_matched.csv')
M_SUMMARY['old labels'].to_csv(tables / 'comparators_matched_old_labels.csv')
QUOTED.to_csv(tables / 'comparators_quoted_own_units.csv')
pd.DataFrame({'unit': d['ids'][M_rows], 'old_structure_id': old_ids.iloc[M_rows].astype(str).to_numpy(),
              'specimen': d['specimen'][M_rows], 'segment': d['segment'][M_rows],
              'old_segment': old_labels}).to_csv(tables / 'unit_set_M.csv', index=False)
unmatched = np.setdiff1d(np.arange(len(d['ids'])), M_rows)
M_COVERAGE = pd.DataFrame({'matched': pd.Series(d['specimen'][M_rows]).value_counts(),
                           'unmatched': pd.Series(d['specimen'][unmatched]).value_counts()}).fillna(0).astype(int)
M_COVERAGE['matched share'] = M_COVERAGE.matched / M_COVERAGE.sum(axis=1)
M_COVERAGE['median log library, matched'] = [np.median(d['log_library'][M_rows][d['specimen'][M_rows] == n]) for n in M_COVERAGE.index]
M_COVERAGE['median log library, unmatched'] = [np.median(d['log_library'][unmatched][d['specimen'][unmatched] == n])
                                              if (d['specimen'][unmatched] == n).any() else np.nan for n in M_COVERAGE.index]
M_COVERAGE.to_csv(tables / 'matched_set_coverage.csv')
old_without = int(len(saved_ids) - len(set(old_ids.iloc[M_rows])))
print(f'M: {len(M_rows):,} of {len(d["ids"]):,} package PT units; {old_without:,} SCF13 PT structures have no package PT unit')
display(M_COVERAGE.round(3))
display(M_SUMMARY['package labels'].round(3))
display(QUOTED)

# AKI: the mouse-only v5 PT DPT, full data only (descriptive; it has no refits).
v5 = ad.read_h5ad(results_root / 'mouse_only_v5' / 'all_mouse_tubules_scanpy_dpt.h5ad', backed='r')
v5_dpt = v5.obs['pt_subset_scanpy_dpt'].astype(float)
v5.file.close()
a = D['aki']
old_aki = UNITS['aki'].obs.old_structure_id.astype(object)
v5_values = v5_dpt.reindex(old_aki.astype(str)).to_numpy(float)
MA_rows = np.flatnonzero(old_aki.notna().to_numpy() & np.isfinite(v5_values))
CTX_MA = context('aki', MA_rows, label='M (AKI)')
AKI_ROWS = []
for arm, z in (('package', a['raw'][MA_rows]), ('package DPT', a['dpt'][MA_rows]), ('v5 PT DPT', v5_values[MA_rows])):
    AKI_ROWS += p1_rows(CTX_MA, species_r(CTX_MA, CTX_MA['EX'], z, CTX_MA['genes']), f'{arm} (full data, not fold-protected)')
AKI_M_P1 = pd.DataFrame(AKI_ROWS)
AKI_M_P1W = pd.DataFrame(sum([p1w_rows(CTX_MA, within_r(CTX_MA, CTX_MA['EX'], z, CTX_MA['genes']), f'{arm} (full data)')
                              for arm, z in (('package', a['raw'][MA_rows]), ('package DPT', a['dpt'][MA_rows]),
                                             ('v5 PT DPT', v5_values[MA_rows]))], []))
AKI_M = pd.concat([AKI_M_P1[AKI_M_P1.primary].pivot_table(index='arm', columns='group', values='rho').add_prefix('P1 '),
                   AKI_M_P1W[AKI_M_P1W.primary].groupby(['arm', 'group']).rho.mean().unstack().add_prefix('P1w ')], axis=1)
AKI_M.to_csv(tables / 'comparators_matched_aki.csv')
print(f'M (AKI): {len(MA_rows):,} of {len(a["ids"]):,} package PT units')
display(AKI_M.round(3))

## 17 · Notebook 71's pass rule (C10) and the criteria

Human vs mouse rows are validation criteria: met or not met, with the claim each allows; none can un-adopt the
package. Relative criteria (C2, C3) compare with SCF13 and DPT13 on M. AKI values are descriptive, shown beside the
same thresholds.

In [ ]:
method71 = nb71 / 'tables' / 'method_table.csv'
C10 = None
if method71.is_file():
    mt = pd.read_csv(method71)
    hit = mt[mt.dataset.astype(str).str.contains('Human', case=False)
             & mt.method.astype(str).str.startswith('normalized') & ~mt.method.astype(str).str.startswith('69')]
    if len(hit):
        C10 = hit.iloc[0].to_dict()
print('Notebook 71 pass rule (C10):', C10 if C10 is not None else 'pending (notebook 71 tables not written yet)')

S = SUMMARY['hvm']['package']
SD_ = SUMMARY['hvm']['package DPT']
MS = M_SUMMARY['package labels']
late_ok = LATE[LATE.dataset.eq('hvm') & LATE.specimen.isin(GROUPS['hvm']['mouse'])]['Spearman with position'].gt(0).all()
trans_ok = bool(wide.loc['hvm']['ordered inside the support'].all())
p2 = P2.set_index(['dataset', 'arm'])
g6 = G6_TABLE.set_index('arm')
calls_a, calls_b = g6.loc['a · refit on half A (independent)', 'pathway calls'], g6.loc['b · base coordinate', 'pathway calls']
noise_pkg = NOISE[(NOISE.context == 'hvm: all PT units') & (NOISE.arm == 'package')]
noise_cells = noise_pkg[noise_pkg.group.isin(['mouse', 'human']) & noise_pkg.specimen.isin(sum(GROUPS['hvm'].values(), []))]
noise_dominated = int((noise_cells['fold noise ÷ spread'] >= 1).sum())
p4h = P4[(P4.context == 'hvm: all PT units') & (P4.arm == 'package')]
p4c_cells = {g: int((p4h[p4h.group.eq(g)].P4c > 0).sum()) for g in ('mouse', 'human')}
p4s = P4[(P4.context == 'hvm: common-support units (sensitivity)') & (P4.arm == 'package')]
a3_frac = A3['fraction human amplitude < mouse (coordinate, NB)']
a3_pct = A3['median notebook-71 T_spatial percentile']


def verdict(value):
    return 'pending' if value is None else ('met' if value else 'not met')


CRITERIA = [
    ('C1', 'Canonical order', 'transitions ordered inside the support in every specimen; Slc7a13 and Slc22a7 rise in both mice',
     f"transitions ordered: {trans_ok}; late markers rise in both mice: {bool(late_ok)}", trans_ok and bool(late_ok)),
    ('C2', 'P1, fold-protected (on M)', '≥ SCF13 − 0.03 in each species',
     f"mouse {MS.loc['P1 mouse', 'package']:.3f} vs SCF13 {MS.loc['P1 mouse', 'SCF13']:.3f}; human {MS.loc['P1 human', 'package']:.3f} "
     f"vs {MS.loc['P1 human', 'SCF13']:.3f} (all units: {S['P1 mouse']:.3f} / {S['P1 human']:.3f}; labels {SUMMARY['hvm'].loc['P1 mouse', 'segment labels']:.3f} / {SUMMARY['hvm'].loc['P1 human', 'segment labels']:.3f})",
     bool(MS.loc['P1 mouse', 'package'] >= MS.loc['P1 mouse', 'SCF13'] - .03 and MS.loc['P1 human', 'package'] >= MS.loc['P1 human', 'SCF13'] - .03)),
    ('C3', 'P1w (on M)', '≥ DPT13 + 0.05 mouse, ≥ DPT13 human',
     f"mouse {MS.loc['P1w mouse', 'package']:.3f} vs DPT13 {MS.loc['P1w mouse', 'DPT13']:.3f}; human {MS.loc['P1w human', 'package']:.3f} "
     f"vs {MS.loc['P1w human', 'DPT13']:.3f} (all units: {S['P1w mouse']:.3f} / {S['P1w human']:.3f})",
     bool(MS.loc['P1w mouse', 'package'] >= MS.loc['P1w mouse', 'DPT13'] + .05 and MS.loc['P1w human', 'package'] >= MS.loc['P1w human', 'DPT13'])),
    ('C4', 'Gene-fold agreement', '≥ 0.70 mouse, ≥ 0.59 human', f"mouse {S['gene-fold mouse']:.3f}; human {S['gene-fold human']:.3f}",
     bool(S['gene-fold mouse'] >= .70 and S['gene-fold human'] >= .59)),
    ('C5', 'LOSO, curve-only', '≥ 0.85', f"{S['LOSO']:.3f}", bool(S['LOSO'] >= .85)),
    ('C6', 'P5, equal depth', '≥ 0.80 overall, ≥ 0.70 human S1', f"overall {S['P5']:.3f}; human S1 {S['P5 human S1']:.3f}",
     bool(S['P5'] >= .80 and S['P5 human S1'] >= .70)),
    ('C7', 'Human S1 resolved', 'gene-fold ≥ 0.50 and P5 ≥ 0.70 in human S1',
     f"gene-fold {S['gene-fold human S1']:.3f}; P5 {S['P5 human S1']:.3f}", bool(S['gene-fold human S1'] >= .50 and S['P5 human S1'] >= .70)),
    ('C8', 'Registration gap', '≤ 0.15 (max over fold and half refits)', f"{S['registration gap']:.3f} (full fit {S['gap of the full fit']:.3f})",
     bool(S['registration gap'] <= .15)),
    ('C9', 'P2 absorption', 'median ratio ≥ 0.85 and human within-segment ρ ≥ 0.90',
     f"ratio {p2.loc[('hvm', 'package'), 'median_ratio']:.3f}; ρ {p2.loc[('hvm', 'package'), 'case within-segment rho']:.3f}",
     not bool(p2.loc[('hvm', 'package'), 'material absorption (notebook 35 flag)'])),
    ('C10', "Downstream gate (notebook 71)", 'NCDR < 0.25 and each relabeling < 5% of tested sets',
     'pending' if C10 is None else f"calls {C10.get('calls')}; relabeled {C10.get('relabeled calls')}; NCDR {C10.get('NCDR')}; rule {C10.get('relabeling rule')}",
     None if C10 is None else str(C10.get('relabeling rule')).strip().lower() == 'pass'),
    ('C11', 'Positive controls (descriptive)', 'sign agreement beside labels',
     '; '.join(f"{r.arm} {r['sign agreement']:.3f}" for _, r in CONTROLS[CONTROLS.set.eq('all positive controls')].iterrows()), None),
    ('C12', 'A3: no attenuation flag', 'flag if ≥ 70% human amplitude < mouse or median 71 T_spatial percentile ≥ 0.75',
     f"fraction {a3_frac:.3f} (labels {A3['fraction human amplitude < mouse (segment labels)']:.3f}, external "
     f"{A3['fraction |Δ human| < |Δ mouse| (external snRNA)']:.3f}); percentile {a3_pct:.3f}",
     None if (np.isnan(a3_pct) and a3_frac < .70) else not (a3_frac >= .70 or a3_pct >= .75)),
    ('C13', 'G6 count split', "not met if arm a's calls < 0.5 × arm b's",
     f"calls a {calls_a}, b {calls_b}, c {g6.loc['c · refit on half B (circular)', 'pathway calls']}; median F c/a "
     f"{g6.loc['c · refit on half B (circular)', 'median F ÷ arm a']:.3f}, b/a {g6.loc['b · base coordinate', 'median F ÷ arm a']:.3f}",
     bool(calls_a >= .5 * calls_b)),
    ('C14', 'Noise budget', 'not met if ≥ 6 of 12 cells have fold noise ÷ spread ≥ 1',
     f"{noise_dominated} of {len(noise_cells)} cells noise-dominated; median ratio {noise_cells['fold noise ÷ spread'].median():.3f}",
     bool(noise_dominated < 6)),
    ('C15', 'P4c claim gate', 'P4c > 0 in ≥ 5/6 mouse and ≥ 4/6 human cells',
     f"mouse {p4c_cells['mouse']}/6; human {p4c_cells['human']}/6; median P4c {p4h[p4h.group.eq('mouse')].P4c.median():.4f} / "
     f"{p4h[p4h.group.eq('human')].P4c.median():.4f}; common-support sensitivity "
     f"{int((p4s[p4s.group.eq('mouse')].P4c > 0).sum())}/6, {int((p4s[p4s.group.eq('human')].P4c > 0).sum())}/6",
     bool(p4c_cells['mouse'] >= 5 and p4c_cells['human'] >= 4)),
]
CRITERIA_TABLE = pd.DataFrame([{'criterion': c_, 'name': n_, 'rule': r_, 'value': v_,
                                'verdict': 'descriptive' if ok is None and c_ == 'C11' else verdict(ok)}
                               for c_, n_, r_, v_, ok in CRITERIA])
CRITERIA_TABLE.to_csv(tables / 'criteria_hvm.csv', index=False)
display(CRITERIA_TABLE)

SA = SUMMARY['aki']['package']
AKI_ROWS = [
    ('C1', 'transitions ordered in every specimen', bool(wide.loc['aki']['ordered inside the support'].all())),
    ('C2', 'P1 control / IR (bar: SCF13 own units 0.745 − 0.03)', f"{SA['P1 control']:.3f} / {SA['P1 IR']:.3f}"),
    ('C3', 'P1w control / IR (bar: DPT13 own units 0.228 + 0.05)', f"{SA['P1w control']:.3f} / {SA['P1w IR']:.3f}"),
    ('C4', 'gene-fold control / IR (bar 0.70)', f"{SA['gene-fold control']:.3f} / {SA['gene-fold IR']:.3f}"),
    ('C5', 'LOSO (bar 0.85)', f"{SA['LOSO']:.3f}"),
    ('C6', 'P5 overall (bar 0.80)', f"{SA['P5']:.3f}"),
    ('C8', 'IR − control gap, max over refits (descriptive)', f"{SA['registration gap']:.3f}"),
    ('C9', 'P2: median ratio; IR within-segment ρ (bars 0.85, 0.90)',
     f"{p2.loc[('aki', 'package'), 'median_ratio']:.3f}; {p2.loc[('aki', 'package'), 'case within-segment rho']:.3f}"),
    ('C14', 'noise-dominated cells', f"{int((NOISE[(NOISE.context == 'aki: all PT units') & (NOISE.arm == 'package')]['fold noise ÷ spread'] >= 1).sum())} of 12"),
    ('C15', 'P4c > 0 cells control / IR',
     f"{int((P4[(P4.arm == 'package') & P4.group.eq('control')].P4c > 0).sum())}/6 / {int((P4[(P4.arm == 'package') & P4.group.eq('IR')].P4c > 0).sum())}/6")]
AKI_TABLE = pd.DataFrame(AKI_ROWS, columns=['criterion', 'measure (threshold from human vs mouse)', 'value'])
AKI_TABLE.to_csv(tables / 'criteria_aki.csv', index=False)
display(AKI_TABLE)

# Long tables.
for name, frame in (('p1', pd.concat([P1, M_P1, AKI_M_P1])), ('p1w', pd.concat([P1W, M_P1W, AKI_M_P1W])),
                    ('within_segment_cells', pd.concat([CELLS, M_CELLS])), ('registration', pd.concat([GAPS, M_GAPS])),
                    ('noise_budget', pd.concat([NOISE, M_NOISE])), ('p4_p4c', pd.concat([P4, M_P4]))):
    frame.to_csv(tables / f'{name}.csv', index=False)
for ds in DATASETS:
    SUMMARY[ds].to_csv(tables / f'summary_{ds}.csv')

## 18 · Figures

In [ ]:
plt.rcParams.update({'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 7, 'axes.titlesize': 7.5,
                     'axes.titleweight': 'bold', 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
                     'legend.fontsize': 6.5, 'legend.frameon': False, 'axes.spines.top': False,
                     'axes.spines.right': False, 'pdf.fonttype': 42, 'savefig.dpi': 300})
GROUP_COLOR = {'mouse': '#0072B2', 'human': '#D55E00', 'control': '#0072B2', 'IR': '#CC79A7'}
SEG_COLOR = {'PT-S1': '#d9d9d9', 'PT-S2': '#969696', 'PT-S3': '#525252'}


def save(fig, name):
    for ext_ in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{ext_}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


for ds in DATASETS:
    p = MARKER_CURVES[ds]
    fig, axes = plt.subplots(1, 6, figsize=(12, 2.2), sharex=True, layout='constrained')
    ref, case = list(GROUPS[ds])
    for i, (ax, gene) in enumerate(zip(axes, MARKERS)):
        for g, key in ((ref, 'reference'), (case, 'case')):
            ax.plot(p['grid'], p[key][i] + np.log(1e4), color=GROUP_COLOR[g], lw=1.6, label=g)
            for n in GROUPS[ds][g]:
                bm = BIN_MEANS[(BIN_MEANS.dataset == ds) & (BIN_MEANS.specimen == n) & (BIN_MEANS.gene == gene)]
                ax.scatter(bm.position, bm['log CP10k'], s=5, color=GROUP_COLOR[g],
                           marker='o' if n == GROUPS[ds][g][0] else 's', alpha=.6, linewidths=0)
        ax.set_title(gene + ('' if gene == 'Slc34a1' else ' (panel/anchor)'), fontsize=6.5)
        ax.set_xlabel('position (71 scale)')
    axes[0].set_ylabel('log CP10k (NB curve; dots = specimen bins)')
    axes[0].legend(loc='best')
    save(fig, f'fig2a_canonical_markers_{ds}')

fig, axes = plt.subplots(2, 4, figsize=(11, 4), sharex=True, sharey=True, layout='constrained')
for row, ds in enumerate(DATASETS):
    for ax, n in zip(axes[row], sum(GROUPS[ds].values(), [])):
        comp = COMPOSITION[(COMPOSITION.dataset == ds) & (COMPOSITION.specimen == n)]
        bottom = np.zeros(len(comp))
        for s in SEGMENTS:
            ax.bar(comp.bin_mid, comp[s], width=.05, bottom=bottom, color=SEG_COLOR[s], label=s, linewidth=0)
            bottom += comp[s].to_numpy()
        for _, t in TRANSITIONS[(TRANSITIONS.dataset == ds) & (TRANSITIONS.specimen == n)].iterrows():
            ax.axvline(t.position, color='#d55e00', lw=1, ls='--')
        ax.set_title(f'{ds} · {n}')
axes[0, 0].legend(loc='lower left')
axes[1, 0].set_xlabel('position (71 scale)')
axes[0, 0].set_ylabel('label share')
save(fig, 'fig2b_composition_transitions')

fig, axes = plt.subplots(1, 2, figsize=(8, 3), layout='constrained')
c = CTX['hvm']
R_pkg = protected(c, pkg_folds, species_r)
for ax, (g, ref) in zip(axes, (('mouse', 'mouse_sn_male'), ('human', 'human_sn_halfB'))):
    mask = evaluation_mask('hvm', g, ref).reindex(c['genes']).to_numpy(bool)
    x, y = contrast[ref].reindex(c['genes'][mask]).to_numpy(), R_pkg.loc[mask, g].to_numpy()
    ax.scatter(x, y, s=2, color=GROUP_COLOR[g], alpha=.3, rasterized=True, linewidths=0)
    ax.axhline(0, color='#999', lw=.6)
    ax.axvline(0, color='#999', lw=.6)
    rho = P1[(P1.arm == 'package') & (P1.group == g) & (P1.reference == ref)].rho.iloc[0]
    ax.set(xlabel=f'external S3 − early ({ref})', ylabel='within-specimen Spearman with the coordinate (fold-protected)',
           title=f'c · P1 {g}: ρ = {rho:.2f}')
save(fig, 'fig2c_p1_external')

order = ['package', 'package DPT', 'SCF13', 'DPT13', 'LCP', 'LRS-Z']
metrics_d = [('P1w mouse', 'P1w, mouse'), ('P1w human', 'P1w, human'), ('gene-fold mouse', 'gene-fold, mouse'),
             ('gene-fold human', 'gene-fold, human'), ('gene-fold human S1', 'gene-fold, human S1'), ('LOSO', 'LOSO')]
fig, axes = plt.subplots(1, len(metrics_d), figsize=(13, 2.4), sharey=False, layout='constrained')
for ax, (key, title) in zip(axes, metrics_d):
    values = [MS.loc[key, a] if key in MS.index else np.nan for a in order]
    ax.bar(range(len(order)), values, color=['#2a78d6', '#7fb2ea', '#999999', '#bbbbbb', '#eda100', '#1baf7a'])
    if key.startswith('P1w'):
        ref_ = M_SUMMARY['package labels'].loc[key, 'read depth (nuisance reference)'] \
            if 'read depth (nuisance reference)' in M_SUMMARY['package labels'] else np.nan
    ax.set_xticks(range(len(order)), order, rotation=60, ha='right')
    ax.axhline(0, color='#555', lw=.6)
    ax.set_title(title)
fig.suptitle('d · Within-segment order on the matched set M (package labels)', fontsize=7.5, x=.01, ha='left')
save(fig, 'fig2d_within_segment_order')

fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), layout='constrained')
names_g6 = list(G6_TABLE.arm)
axes[0].bar(range(3), G6_TABLE['pathway calls'], color=['#2a78d6', '#999999', '#d55e00'])
axes[0].set_xticks(range(3), [n.split(' · ')[0] for n in names_g6])
axes[0].set(ylabel='cameraPR calls on half B', title='e · Count split: a independent, b base, c circular')
pp = P2.set_index(['dataset', 'arm'])
labels_p2 = [f'{ds} {arm}' for ds, arm in pp.index]
axes[1].bar(range(len(pp)), pp['median_ratio'], color='#2a78d6')
axes[1].axhline(.85, color='#d55e00', lw=1, ls='--')
axes[1].set_xticks(range(len(pp)), labels_p2, rotation=30, ha='right')
axes[1].set(ylabel='P2 median ratio (1 = not absorbed)', title='e · Injected group × position effect')
save(fig, 'fig2e_robustness')

fig, ax = plt.subplots(figsize=(7, 4.6), layout='constrained')
rows_s4 = ['P1 mouse', 'P1 human', 'P1w mouse', 'P1w human', 'gene-fold mouse', 'gene-fold human', 'gene-fold human S1',
           'P5', 'P5 human S1', 'count split A vs B', 'registration gap', 'LOSO', 'injection: human within-segment ρ',
           'median fold noise ÷ spread', 'P4c mean mouse', 'P4c mean human']
heat = MS.reindex(index=rows_s4, columns=order).astype(float)
ax.imshow(np.zeros_like(heat.to_numpy()), cmap='Greys', vmin=0, vmax=1)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.iat[i, j]
        ax.text(j, i, '–' if not np.isfinite(v) else (f'{v:.3f}' if abs(v) < .1 else f'{v:.2f}'), ha='center', va='center', fontsize=6)
ax.set_xticks(range(len(order)), order)
ax.set_yticks(range(len(rows_s4)), rows_s4)
ax.set_title('S4 · Head-to-head on the matched set M (package labels); quoted own-unit values in the tables')
save(fig, 'figS4_head_to_head')

READING.append('A refit that fails the orientation guard enters every metric as NaN (notebook 36). Gene-fold agreement, '
               'fold-protected P1/P1w and positive controls, the gap and the noise budget then use the remaining refits '
               '(medians and maxima ignore NaN, as notebooks 36 and 51); its P4/P4c cells count as not positive.')
READING.append("P1's label reference is notebook 51's (labels coded 0/1/2 used as a coordinate); the positive controls' label "
               "reference is notebook 35's S3 − early contrast.")
READING.append('Scale-dependent metrics (gap, noise) rescale every coordinate by notebook 71\'s rule within the units of the '
               'context, so the gap on all PT units and on M can differ; both are reported, C8 uses all PT units.')
READING.append('C12 counts as not met when the attenuation flag is raised; the label-based and external fractions are '
               'shown beside it, since the flag cannot tell positional noise from biology when they agree.')
READING.append("LRS-Z's gap on M uses its three gene folds only (notebook 53 saved no half refits); LCP's LOSO and every "
               "comparator's P2 ratio and downstream gate are quoted from their own units.")

## 19 · Run record

In [ ]:
RUN_RECORD = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'refit_logic_version': REFIT_LOGIC_VERSION, 'package': EXTERNAL,
    'confirmed_pass1_fingerprints': CONFIRMED_PASS1, 'refits': REFIT_RECORD.drop(columns=['root']).to_dict(orient='records'),
    'criteria_hvm': CRITERIA_TABLE.to_dict(orient='records'), 'criteria_aki': AKI_TABLE.to_dict(orient='records'),
    'notebook_71_available': {'method_table': method71.is_file(), 'T_spatial': T71 is not None, 'calls': calls71 is not None},
    'matched_set': {'hvm': int(len(M_rows)), 'aki': int(len(MA_rows))},
    'how_the_protocol_was_read': READING}
(output / 'run_record.json').write_text(json.dumps(RUN_RECORD, indent=1, default=str))
print('How the protocol was read:')
print('\n'.join(f'- {line}' for line in READING))

## 20 · Summary

**Guards.** Both identity refits reproduce notebook 70 bit for bit (pass-1 fingerprints `efafc9ac2e99` and
`bb4a80c1b484`, maximum |Δ| 0). One refit failed the orientation guard: human vs mouse fold 2 runs through the
mouse PT from S3 to S1. The package DPT passed in that refit. Run-to-run sensitivity of the package itself:
- numerical floor: within-segment ρ 0.96 against the base;
- 12 instead of the default threads: ρ 0.98, with a different pass-1 clustering.

**Validation criteria, human vs mouse.** They are met or not met for the package coordinate; none un-adopts it.

| # | Criterion | Value | Verdict |
|---|---|---|---|
| C1 | canonical order | transitions out of order in HUK1_MED1 (S1→S2 0.666 > S2→S3 0.590; human PT-S2 has 259 units); Slc7a13/Slc22a7 rise in both mice (ρ 0.69–0.85) | not met |
| C2 | P1 on M vs SCF13 − 0.03 | mouse 0.746 vs 0.746; human 0.295 vs 0.246 (labels 0.749 / 0.314) | met |
| C3 | P1w on M vs DPT13 (+0.05 mouse) | mouse 0.367 vs 0.288; human 0.156 vs 0.130 | met |
| C4 | gene-fold ≥ 0.70 / 0.59 | mouse 0.751; human 0.488 (fold 2 failed) | not met |
| C5 | LOSO ≥ 0.85 | 0.959 (curve only) | met |
| C6 | P5 ≥ 0.80, human S1 ≥ 0.70 | 0.731; 0.552 | not met |
| C7 | human S1 resolved | gene-fold 0.516; P5 0.552 | not met (human S1 unresolved) |
| C8 | registration gap ≤ 0.15 | 0.369 (half A; 0.217 on M; full fit 0.076) | not met |
| C9 | P2 absorption | ratio 0.985; human ρ 0.922 | met |
| C10 | notebook 71's gate | 83 calls; relabeled 1, 0; NCDR 0.006 | met |
| C11 | positive controls | sign agreement 0.899 (labels 0.959) | descriptive |
| C12 | A3 attenuation | flag raised: 0.829 of H genes smaller in human (labels 0.829, external 0.771); T_spatial percentile 0.943 | not met |
| C13 | count split | calls a 66, b 81, c 66; median F b/a 1.22, c/a 1.06 | met |
| C14 | noise budget | 10 of 12 cells with fold noise ≥ the within-segment spread | not met |
| C15 | P4c claim gate | mouse 1/6, human 0/6 cells > 0 (common-support sensitivity 2/6, 0/6) | not met |

**AKI (descriptive, against the same thresholds).**
- Transitions are ordered in all four specimens.
- P1: control 0.745, IR 0.363. P1w: control 0.300, IR 0.181.
- Gene-fold: control 0.812, IR 0.705. LOSO 0.976. P5 0.890.
- P2: ratio 0.984, IR ρ 0.961. IR − control gap 0.086.
- Noise-dominated cells: 11 of 12. P4c > 0 cells: control 3/6, IR 3/6.

**Comparators on M** (11,488 of 11,947 units, package labels).
- **Against SCF13:** the package is better on human P1, P1w, gene-fold, P5, count split, gap and LOSO. It is equal
  on mouse P1 and on human-S1 P5.
- **Against DPT13:** the package is better on mouse P1w and gene-fold, and worse on human gene-fold and P5.
- **Against LCP and LRS-Z:** both landmark coordinates beat it on every within-segment stability metric:
  - gene-fold 0.96 and 0.93 (mouse), 0.82 and 0.97 (human), against 0.76 and 0.49;
  - count split 0.89 (LCP), against 0.26;
  - noise-dominated cells 0 and 0, against 8.
- No arm clears P4c.

**What the paper may claim.**
- The package coordinate recovers external S3-versus-early zonation as well as the labels and SCF13.
- Its within-segment order agrees with external adjacent contrasts better than SCF13 and DPT13.
- It does not absorb an injected group-by-position effect in either dataset.
- Its positional calls are not created by reusing reads.

**What the paper may not claim.**
- Within-segment positions are not stable to gene panels, read halves or depth.
- Human S1 is unresolved.
- The species placement moves by more than 0.15 between refits.
- Positional noise exceeds the within-segment spread in most cells.
- The coordinate does not predict held-out genes beyond segments plus coverage.

Continuous claims must therefore rest on between-segment structure and on replication across specimens.